# 20 · Economic credit and teacher-guided improvement

## Purpose

Train AHR on positions its own player reaches, using KataGo **only as an offline teacher**. Compare a standard teacher-guided control against bounded opportunity-cost weighting and an auxiliary economic-credit head. Winning remains the objective: terminal win **+1**, loss **−1**, draw **0**. Captures, stone count and passing receive no flat bonuses or penalties.

**Inputs:** a verified notebook 19 player, the compatible notebook 18 teacher cache, pinned KataGo assets, and fresh development rollouts. **Outputs:** a reusable frozen labelled buffer, resumable control/economics checkpoints, logs and comparison-ready metrics. A single buffer supports both arms with identical starting weights, examples, seeds and update counts. Data used here is development data; notebook 17’s consumed games and the dataset test split are not used as fresh evaluation.

**Runtime:** a small CPU-labelled buffer can take tens of minutes; GPU teacher labels are faster when a verified backend is configured. Long runs belong in the project’s detached controller. Checkpoints live in `checkpoints/<buffer_id>-collection/` and `checkpoints/<run_id>/`; latest two valid states plus the best are retained. This notebook performs short validation on Run All until the controller cell is configured.

**Resume:** repeat `go2_k20_guided_improve(PROJECT, saved_config)` with the exact JSON configuration saved under the run checkpoint directory. A `STOP` file requests a graceful pause; remove it explicitly before resuming. Corruption, changed source hashes or a changed teacher/model identity refuse recovery. Only one writer may own each run.


**Run All now launches the complete notebook18–20 workflow in detached tmux after source-bound recovery checks.** Set `AHR_K20_MODE=validate` for disposable checks only, or `status` to inspect progress and play without launching.

## Methods and assumptions

A frozen forecast defines a potential on the **full Go history**, including ko history, rules, komi and player to move. We express it consistently for Black, set terminal potential to zero, and use undiscounted episodic returns. For mover colour $c_t\in\{+1,-1\}$,

$$r'_t=c_t\,[r^{Black}_t+\eta(\Phi^{Black}(s_{t+1})-\Phi^{Black}(s_t))],\quad G'_t=r'_t-G'_{t+1}.$$

For a completed game this telescopes to $G'_t=z_t-\eta\Phi^{mover}(s_t)$. The potential supplies a fixed state baseline, **not new outcome information**. The separate credit head predicts this shaped return; the Worker remains trained on unshaped win/outcome targets. The algebra preserves the terminal objective under the stated boundary conditions; it does not prove neural training convergence or full-game optimality. An incomplete game remains UNKNOWN and has no observed outcome or credit-return target.

The concrete economics change also weights a searched teacher policy by the estimated opportunity cost of AHR’s selected move. Opportunity cost is the teacher’s estimated best-action win probability minus its estimated selected-action win probability, clipped to [0,1]. We cap the multiplier to avoid a few uncertain labels dominating learning. The best and forced-action forecasts are estimates from a frozen teacher, not proofs. No universal risk-aversion bonus is added: search attention can focus on uncertainty, while the objective stays expected winning.

Sources: [Ng, Harada & Russell: potential-based reward shaping](https://people.eecs.berkeley.edu/~russell/papers/icml99-shaping.pdf), [Lu, Schwartz & Givigi: shaping in stochastic games](https://arxiv.org/abs/1401.3907), [DAgger: learning on induced state distributions](https://proceedings.mlr.press/v15/ross11a.html), [KataGo analysis API](https://github.com/lightvector/KataGo/blob/master/docs/Analysis_Engine.md), and [proper forecast scoring](https://sites.stat.washington.edu/people/raftery/Research/PDF/Gneiting2007jasa.pdf).


In [ ]:
from pathlib import Path
import os, nbformat
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
PROJECT=Path.cwd().resolve()
while not (PROJECT/'AGENTS.md').is_file() and PROJECT!=PROJECT.parent: PROJECT=PROJECT.parent
for name in ['00_contracts_and_recovery.ipynb','15_teacher_evaluation_and_launch.ipynb']:
    for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,name+':ahr-export','exec'),globals())
globals().update(go2_teacher_arena_namespace(PROJECT))
for prefix in ('17_','18_','19_'):
    paths=list((PROJECT/'experiments/ahr_v2').glob(prefix+'*.ipynb'))
    if len(paths)!=1: raise RuntimeError('One canonical upstream notebook required: '+prefix)
    go2_import_exports(paths[0],globals())
from IPython.display import display, Markdown
import pandas as pd
torch.set_num_threads(4)


### 1. Fixed objective, potentials and bounded opportunity cost

In [ ]:
GO2_K20_BOOK='20_economic_rewards_and_teacher_guided_improvement.ipynb'

def go2_k20_reward_config(run_id='ahr-v2-katago-guided-economics-v1',base_run_id=None,
                         cache_id='go9x9-competitive-v2',device='cuda:1',arm='economics',
                         buffer_id='ahr-v2-guided-buffer-v1',base_which='best'):
    if arm not in ('control','economics'): raise ValueError('Arm must be control or economics.')
    return {'schema':'ahr-go-v2-economic-guided-v1','run_id':run_id,'base_run_id':base_run_id,
        'cache_id':cache_id,'device':device,'arm':arm,'buffer_id':buffer_id,'base_which':base_which,'seed':2026101120,
        'purpose':'development','board_size':9,'komi':7.5,'games':8,'rows_per_game':3,
        'opponent_cycle':['self','katago-v1'],'opponent_timeout_seconds':90,
        'exclude_benchmark_run_ids':['ahr-v2-benchmark-20261009-v1'],
        'max_plies':400,'game_time_seconds':900,'opening_plies':8,'opening_temperature':1.5,
        'root_noise_fraction':0.15,'simulations':128,'search_batch_size':8,
        'search_wall_seconds':None,'reading_nodes':128,'reading_wall_seconds':0.02,
        'teacher_visits':64,'teacher_timeout_seconds':120,'teacher_backend':'cpu','teacher_gpu_device':0,
        'potential_strength':1.0,'gamma':1.0,'regret_scale':0.20,'regret_multiplier_cap':3.0,
        'regret_weight':1.0 if arm=='economics' else 0.0,
        'credit_loss_weight':0.10 if arm=='economics' else 0.0,
        'batch_size':128,'anchor_fraction':0.5,'optimizer_steps':128,'learning_rate':0.00010,
        'weight_decay':0.00010,'gradient_clip':2.0,'cpu_threads':4,
        'precision':'bfloat16' if str(device).startswith('cuda') else 'float32',
        'checkpoint_seconds':120,'checkpoint_events':250,'checkpoint_optimizer_steps':100,
        'evaluate_every_steps':50,'development_rows':512,'time_budget_hours':8.0,
        'loss_weights':{'policy':1.0,'teacher_value':1.0,'game_outcome':0.5,'score':0.5,'ownership':0.25}}

def go2_k20_validate_config(config):
    config=copy.deepcopy(config)
    if config.get('schema')!='ahr-go-v2-economic-guided-v1': raise ValueError('Unsupported guided configuration.')
    for key in ('run_id','buffer_id'):
        if Path(config[key]).name!=config[key] or not config[key].startswith('ahr-v2-'):
            raise ValueError('Local ahr-v2 run identifiers required.')
    if config['base_which'] not in ('best','latest'): raise ValueError('An explicit best/latest base checkpoint is required.')
    if config['purpose'] not in ('development','validation_fixture'): raise ValueError('Development scope required.')
    if config['arm'] not in ('control','economics') or config['board_size']!=9 or config['komi']!=7.5 or config['gamma']!=1.0:
        raise ValueError('A 9×9 undiscounted control/economics experiment is required.')
    for key in ('games','rows_per_game','max_plies','simulations','search_batch_size','teacher_visits',
                'batch_size','optimizer_steps','evaluate_every_steps','development_rows'):
        if type(config[key]) is not int or config[key]<1: raise ValueError('Positive integer required: '+key)
    if type(config['opening_plies']) is not int or not 0<=config['opening_plies']<=16 or not 0<config['opening_temperature']<=4:
        raise ValueError('Invalid bounded opening parameters.')
    if not 0<=config['root_noise_fraction']<=1 or type(config['teacher_gpu_device']) is not int or config['teacher_gpu_device']<0:
        raise ValueError('Invalid rollout exploration or explicit teacher GPU device.')
    for key in ('potential_strength','regret_weight','credit_loss_weight'):
        if not math.isfinite(config[key]) or config[key]<0: raise ValueError('Nonnegative finite coefficient required: '+key)
    if config['potential_strength']>1 or not 0<config['anchor_fraction']<1:
        raise ValueError('Bounded potential strength and mixed anchor rows required.')
    if config['regret_scale']<=0 or not 1<=config['regret_multiplier_cap']<=10:
        raise ValueError('Positive regret scale and capped policy weighting required.')
    if not 0<config['checkpoint_seconds']<=900 or not 1<=config['checkpoint_events']<=250:
        raise ValueError('Checkpoints must obey the project time/event limits.')
    if not 1<=config['checkpoint_optimizer_steps']<=250: raise ValueError('Invalid optimizer checkpoint cadence.')
    if not config['opponent_cycle'] or any(x not in ('self','katago-v1','katago-v4','katago-v16') for x in config['opponent_cycle']):
        raise ValueError('Supported frozen rollout opponent profiles required.')
    if config['arm']=='control' and (config['regret_weight'] or config['credit_loss_weight']):
        raise ValueError('The control must disable economic additions.')
    return config

def go2_k20_potential_reward(mover,potential_black,next_potential_black,*,terminal=False,winner=None,
                            strength=1.0,gamma=1.0):
    if mover not in (-1,1) or gamma!=1.0 or not 0<=strength<=1: raise ValueError('Undiscounted alternating zero-sum reward required.')
    if any(not math.isfinite(float(x)) or abs(float(x))>1.000001 for x in (potential_black,next_potential_black)):
        raise ValueError('Finite frozen forecast potentials in [-1,1] are required.')
    if terminal and (winner not in (-1,0,1) or abs(next_potential_black)>1e-12):
        raise ValueError('A terminal reward requires its actual winner and zero terminal potential.')
    if not terminal and winner is not None: raise ValueError('No observed winner before termination.')
    return float(mover*((winner if terminal else 0)+strength*(next_potential_black-potential_black)))

def go2_k20_potential_returns(movers,potentials_black,*,winner,strength=1.0,gamma=1.0):
    if winner not in (-1,0,1): raise ValueError('UNKNOWN trajectories have no observed return target.')
    if len(potentials_black)!=len(movers)+1 or not movers: raise ValueError('One potential per full-history state including the terminal state is required.')
    if any(movers[i+1]!=-movers[i] for i in range(len(movers)-1)): raise ValueError('Mover colours must alternate.')
    rewards=[go2_k20_potential_reward(c,potentials_black[i],potentials_black[i+1],
        terminal=i==len(movers)-1,winner=winner if i==len(movers)-1 else None,
        strength=strength,gamma=gamma) for i,c in enumerate(movers)]
    returns=[0.0]*len(rewards);following=0.0
    for i in range(len(rewards)-1,-1,-1):
        following=rewards[i]-gamma*following;returns[i]=following
    expected=[float(c*(winner-strength*potentials_black[i])) for i,c in enumerate(movers)]
    if not np.allclose(returns,expected,atol=1e-12,rtol=0): raise RuntimeError('Zero-sum telescoping failed.')
    return {'rewards':rewards,'returns':returns,'unshaped_returns':[int(c*winner) for c in movers],
            'fixed_state_baselines':[float(strength*c*potentials_black[i]) for i,c in enumerate(movers)],
            'perspective':'mover','gamma':gamma,'terminal_potential':0.0}

def go2_k20_opportunity_cost(best_value,selected_value):
    if any(not math.isfinite(float(x)) or abs(float(x))>1.00001 for x in (best_value,selected_value)):
        raise ValueError('Teacher action values must be signed win estimates in [-1,1].')
    return float(np.clip((best_value-selected_value)/2,0,1))

def go2_k20_regret_weight(regret,config):
    regret=np.asarray(regret,dtype=np.float64)
    if not np.isfinite(regret).all() or (regret<0).any() or (regret>1).any(): raise ValueError('Invalid opportunity-cost estimates.')
    return np.minimum(config['regret_multiplier_cap'],1+config['regret_weight']*regret/config['regret_scale'])

def go2_k20_sources(project):
    result=go2_bench_sources(project)
    for prefix in ('18_','19_','20_'):
        paths=list((Path(project)/GO2_ROOT).glob(prefix+'*.ipynb'))
        if len(paths)!=1: raise RuntimeError('One canonical notebook required: '+prefix)
        result[str(paths[0].relative_to(project))]=go2_code_sha(paths[0])
    return result


def go2_k20_opening(actor,generator,config,used_families):
    if config['opening_plies']==0:
        return go2_initial_state(9,config['komi']),[],go2_teacher_opening_family([])
    if not 1<=config['opening_plies']<=16 or not 0<config['opening_temperature']<=4:
        raise ValueError('A bounded forced opening and positive temperature are required.')
    for attempt in range(200):
        position=go2_initial_state(9,config['komi']);trace=[]
        for _ in range(config['opening_plies']):
            output=go2_eval_batch(actor,[position])[0];policy=np.asarray(output['prior'],dtype=float)
            # Forced opening books contain board moves. Once actual play starts, passing remains legal.
            policy[-1]=0.0;policy=np.power(np.maximum(policy,0),1/config['opening_temperature'])
            if policy.sum()<=0:raise RuntimeError('No compatible forced opening move.')
            policy/=policy.sum();action=int(torch.multinomial(torch.tensor(policy,dtype=torch.float64),1,generator=generator))
            trace.append({'kind':'play','color':position.to_play,'action':action,'opening_forced':True,'ahr_move':False})
            position=go2_step(position,action)
        family=go2_teacher_opening_family(position.actions)
        if family not in used_families:return position,trace,family
    raise RuntimeError('Fresh rollout opening families exhausted; collection refused.')


def go2_k20_opening_exclusions(project,config):
    project=Path(project);families=set();inputs=[]
    old=go2_teacher_dataset(project,'go9x9-teacher-v1',verify=False)
    new=go2_k18_dataset(project,cache_id=config['cache_id'],verify=False)
    for name,dataset in [('original_teacher',old),('competitive_teacher',new)]:
        manifest=dataset['manifest']
        for split in ('train','dev'):families.update(manifest['families'][split])
        inputs.append({'kind':name,'manifest_sha256':go2_digest(manifest),'cache_path':dataset.get('cache_path')})
    # Original teacher development reviews are consumed evidence, not new final data.
    for path in sorted((project/'reports/ahr_v2/evaluations').glob('*/report.json')):
        record=json.loads(path.read_text());accepted=[]
        for game in record.get('games',[]):
            actions=game.get('state',{}).get('actions',[])
            if len(actions)>=8:accepted.append(go2_teacher_opening_family(actions[:8]))
        if accepted:
            families.update(accepted);inputs.append({'file':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    required_paths=[project/'reports/ahr_v2/benchmarks'/run_id/'plan.json' for run_id in config['exclude_benchmark_run_ids']]
    for path in required_paths:
        if not path.is_file():raise FileNotFoundError('Consumed benchmark plan required: '+str(path))
    # Freeze every currently existing screen/confirmation plan before collecting any new training game.
    plan_paths=sorted(set((project/'reports/ahr_v2/benchmarks').glob('*/plan.json'))|set(required_paths))
    for path in plan_paths:
        plan=json.loads(path.read_text())
        for game in plan.get('games',[]):
            family=game.get('family_sha',game.get('opening_family'))
            if family:families.add(family)
            actions=game.get('opening_actions',[])
            if len(actions)>=8:families.add(go2_teacher_opening_family(actions[:8]))
        inputs.append({'file':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    return {'families':sorted(families),'inputs':inputs,'counts':{'excluded_families':len(families)},
            'exclusion_scope':'original and competitive teacher train/dev; consumed teacher reviews and every benchmark plan present at buffer creation',
            'frozen_inventory':True}

def go2_k20_select_rows(candidates,count,generator):
    """Approximately half critical contexts, half uniform other turns; no outcome-based selection."""
    if not candidates:return []
    conditions=[];priority=[]
    for index,row in enumerate(candidates):
        state=go2_state_from_dict(row['state']);successor=go2_state_from_dict(row['next_state']);reason=[]
        if row['action']==state.pass_action or state.consecutive_passes:reason.append('pass_decision')
        if any(len(liberties)<=1 for stones,liberties in go2_groups(state.board,9)):reason.append('group_in_atari')
        opponent=-state.to_play
        if state.board.count(opponent)>successor.board.count(opponent):reason.append('capture_transition')
        if state.move_number>=60:reason.append('late_cleanup_context')
        conditions.append(reason)
        if reason:priority.append(index)
    total=min(count,len(candidates));priority_count=min((total+1)//2,len(priority))
    selected=[]
    if priority_count:
        order=torch.randperm(len(priority),generator=generator)[:priority_count].tolist()
        selected=[{'index':priority[k],'sampling_reason':'critical_context','conditions':conditions[priority[k]]} for k in order]
    used={x['index'] for x in selected};other=[i for i in range(len(candidates)) if i not in used]
    needed=total-len(selected)
    for k in torch.randperm(len(other),generator=generator)[:needed].tolist():
        index=other[k];selected.append({'index':index,'sampling_reason':'uniform_other_turn','conditions':conditions[index]})
    return selected


### 2. Frozen rollouts and targeted offline queries

AHR chooses its moves without teacher inference. Rollouts alternate self-play and a pinned 1-visit opponent; exact opponent command traces and pending AHR reading/search are saved. Only a few deterministic sampled AHR positions per game receive deeper labels. Root and forced selected-move forecasts share the frozen teacher and complete legal history. Incomplete games may still provide labelled policy examples, but never observed winner or credit targets.

The labelled buffer is immutable after completion and reused by both arms. Each label includes query provenance, search-policy validity and its estimated opportunity cost. Rollout termination and local proof statuses remain distinct.


In [ ]:
class _Go2K20Pause(Exception): pass

def _go2_k20_cpu_devices(device):
    return [int(torch.device(device).index or 0)] if str(device).startswith('cuda') else []

def _go2_k20_teacher_pin(project,config):
    if config['teacher_backend']=='cpu':
        assets=go2_bench_katago_assets(project,allow_download=False)
    elif config['teacher_backend']=='cuda':
        factory=globals().get('go2_k20_gpu_assets')
        if factory is None: raise RuntimeError('The explicit CUDA teacher dependency has not been pinned.')
        assets=factory(project)
    else: raise ValueError('Explicit cpu or cuda teacher backend required.')
    return {'assets':copy.deepcopy(assets),'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),
            'board_size':9,'komi':config['komi'],'visits':config['teacher_visits'],
            'teacher_gpu_device':config['teacher_gpu_device'],'stateless_protocol':go2_k20_stateless_teacher_identity(project),'value_kind':'frozen native-pass-alive teacher estimate; not a proved game result'}

def _go2_k20_teacher(project,config):
    if config['teacher_backend']=='cuda':
        factory=globals().get('go2_k20_gpu_teacher')
        if factory is None: raise RuntimeError('Explicit CUDA teacher factory missing; backend will not silently change.')
        return factory(project,visits=config['teacher_visits'],timeout_seconds=config['teacher_timeout_seconds'],seed=config['seed'],gpu_device=config['teacher_gpu_device'])
    factory=globals().get('go2_k20_cpu_teacher')
    if factory is None:raise RuntimeError('Stateless CPU teacher factory is missing; persistent teacher recovery is refused.')
    return factory(project,visits=config['teacher_visits'],timeout_seconds=config['teacher_timeout_seconds'],seed=config['seed'])

def _go2_k20_collection_config(config):
    fields=('schema','buffer_id','base_run_id','base_which','cache_id','seed','purpose','board_size','komi','games','device','precision','cpu_threads','teacher_gpu_device',
        'rows_per_game','opponent_cycle','opponent_timeout_seconds','exclude_benchmark_run_ids','max_plies','game_time_seconds',
        'opening_plies','opening_temperature','root_noise_fraction','simulations','search_batch_size','search_wall_seconds','reading_nodes','reading_wall_seconds',
        'teacher_visits','teacher_timeout_seconds','teacher_backend','potential_strength','gamma')
    return {key:copy.deepcopy(config[key]) for key in fields}

def _go2_k20_buffer_report(project,state,status):
    games=state['replay'];rows=state['labelled_rows']
    report={'run_id':state['config']['buffer_id']+'-collection','buffer_id':state['config']['buffer_id'],
        'status':status,'stage':state['stage'],'global_step':state['global_step'],
        'games_finished':len(games),'games_requested':state['config']['games'],
        'unknown_games':sum(x['winner'] is None for x in games),'labelled_rows':len(rows),
        'label_failures':len(state['label_failures']),'teacher_queries':len(state['teacher_cache']),
        'collection_identity':state['identity'],'opponent_used_only_during_rollout':True,
        'teacher_used_only_for_offline_labels':True,'updated_at_utc':go2_now()}
    directory=Path(project)/'reports/ahr_v2'/report['run_id'];directory.mkdir(parents=True,exist_ok=True)
    go2_atomic_json(directory/'status.json',report)
    return report

def _go2_k20_fixture_evaluator(states):
    outputs=[]
    for state in states:
        legal=go2_legal_mask(state);prior=np.zeros(state.size*state.size+1,dtype=float);prior[-1]=1.0
        outputs.append({'prior':prior,'value':0.0,'score':0.0,'ownership':np.zeros(state.size*state.size),
                        'legal_mask':legal,'cache_key':go2_cache_key(state)})
    return outputs

class _Go2K20FixtureTeacher:
    def __init__(self,*args,**kwargs): pass
    def query(self,state,visits=None,allowed_actions=None):
        legal=np.asarray(go2_legal_mask(state),dtype=bool);policy=legal.astype(float)
        if allowed_actions is not None:
            policy[:]=0
            for action in allowed_actions:
                if not legal[action]: raise ValueError('Fixture selected action illegal.')
                policy[action]=1
        else: policy[-1]+=10
        policy/=policy.sum();value=float(0.3 if state.to_play==1 else -0.3)
        if allowed_actions is not None: value-=0.2
        return {'policy':policy,'value':value,'score':0.1,'ownership':np.zeros(81),
                'score_valid':True,'ownership_valid':True,'policy_valid':True,
                'request':{'moves':list(state.actions),'visits':visits,'allowed_actions':allowed_actions},
                'provenance':{'kind':'synthetic validation fixture','native_pass_alive_estimate':True}}
    def close(self): pass

def go2_k20_collect(project,config,max_events=None,on_checkpoint=None,*,_fixture=None):
    """Collect one frozen DAgger iteration. No optimizer updates; no hidden model promotion."""
    project=go2_project(project);config=go2_k20_validate_config(config)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    torch.set_num_threads(config['cpu_threads'])
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
    if _fixture is not None and config['purpose']!='validation_fixture': raise ValueError('Synthetic hooks require validation_fixture purpose.')
    if _fixture is None:
        if not config['base_run_id']: raise ValueError('An explicit verified notebook 19 base run is required.')
        actor,metadata=go2_k19_load_player(project,config['base_run_id'],which=config['base_which'],device=config['device'])
        if config['purpose']=='development' and (metadata.get('is_smoke',False) or metadata.get('optimizer_step',metadata.get('global_step',0))<=0):
            raise RuntimeError('A trained nonsmoke notebook 19 base is required for development collection.')
        teacher_pin=_go2_k20_teacher_pin(project,config)
        opponent_pins={profile:go2_bench_opponent_pin(project,profile) for profile in set(config['opponent_cycle']) if profile!='self'}
    else:
        actor=Go2Network(Go2NetConfig(width=8,blocks=1)).to(config['device']);go2_seed(config['seed'])
        # Fixture initial weights are seeded independently of any caller RNG.
        actor=Go2Network(Go2NetConfig(width=8,blocks=1)).to(config['device'])
        metadata={'run_id':'ahr-v2-k20-fixture-base','checkpoint_file':'synthetic','checkpoint_sha256':'synthetic',
                  'model_identity':go2_model_identity(actor),'network':dataclasses.asdict(actor.config),'is_smoke':True}
        teacher_pin={'kind':'synthetic_validation','schema':'k20-pass-teacher-v1'};opponent_pins={}
        if config['opponent_cycle']!=['self']: raise ValueError('Synthetic validation uses self-play only.')
    actor.eval();actor.requires_grad_(False)
    directory=project/'checkpoints'/(config['buffer_id']+'-collection')
    generator=torch.Generator(device='cpu').manual_seed(config['seed']+20)
    with go2_checkpoint_lock(directory):
        # Validate all retained states before using their immutable exclusion inventory.
        previous,previous_entry=go2_load_checkpoint(directory)
        if previous is None:
            exclusions={'families':[],'inputs':[],'fixture':True,'frozen_inventory':True} if _fixture is not None else go2_k20_opening_exclusions(project,config)
        else:
            exclusions=copy.deepcopy(previous['identity']['opening_exclusions'])
            if not exclusions.get('frozen_inventory'):raise RuntimeError('A frozen opening-exclusion inventory is required; resume refused.')
        identity={'schema':'ahr-go-v2-guided-buffer-v1','sources':go2_k20_sources(project),
            'base_checkpoint':{key:metadata.get(key) for key in ('run_id','checkpoint_file','checkpoint_sha256','model_identity')},
            'collection_config':_go2_k20_collection_config(config),'teacher':teacher_pin,'opponents':opponent_pins,'opening_exclusions':exclusions}
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'model':copy.deepcopy(actor.state_dict()),
                'optimizer':{},'scheduler':{},'scaler':{},'rng':{},'sampler':{'game_index':0,'query_index':0,'opening_families':[]},
                'global_step':0,'config':config,'metrics':{},'stage':'COLLECT','replay':[],
                'pending_rows':[],'labelled_rows':[],'label_failures':[],'teacher_cache':{},
                'current_game':None,'search':None,'data_cursor':{'game_index':0,'query_index':0},
                'distributed':{'kind':'single_writer_single_actor'},'active_seconds':0.0,'model_metadata':metadata}
        else:
            actor.load_state_dict(state['model']);go2_restore_rng(state['rng'],{'collection':generator})
        initial_event=state['global_step'];last_saved=-1 if entry is None else state['global_step'];last_time=time.monotonic()
        last_count=state['global_step'];live_tree=None;opponent=None;teacher=None;game_clock=None;elapsed_base=0.0
        def save(force=False):
            nonlocal entry,last_saved,last_time,last_count
            if not force and time.monotonic()-last_time<config['checkpoint_seconds'] and state['global_step']-last_count<config['checkpoint_events']: return
            if state['global_step']==last_saved: return
            if state['current_game'] is not None and game_clock is not None:
                state['current_game']['elapsed_seconds']=elapsed_base+time.monotonic()-game_clock
            state['search']=go2_search_state_dict(live_tree) if live_tree is not None else None
            state['rng']=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'collection':generator})
            state['metrics']={'games_finished':len(state['replay']),'labelled_rows':len(state['labelled_rows']),
                'queries_finished':len(state['teacher_cache']),'query_failures':len(state['label_failures'])}
            state['data_cursor']=copy.deepcopy(state['sampler'])
            entry=go2_save_checkpoint(directory,state,best=state['stage']=='COMPLETE',
                resume_cell='20: go2_k20_collect(PROJECT,json.loads((PROJECT/"checkpoints"/(buffer_id+"-collection")/"config.json").read_text()))')
            last_saved=state['global_step'];last_count=state['global_step'];last_time=time.monotonic()
            if on_checkpoint is not None:
                rng=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'collection':generator})
                try:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
                finally:go2_restore_rng(rng,{'collection':generator})
        def event(force=False):
            state['global_step']+=1
            if max_events is not None and state['global_step']-initial_event>=max_events:
                save(True);raise _Go2K20Pause()
            save(force)
        def on_batch(tree):
            nonlocal live_tree
            live_tree=tree;event()
        go2_atomic_json(directory/'config.json',state['config'])
        if state['stage']=='COMPLETE': return state
        save(True)
        status='RUNNING'
        try:
            while state['stage']=='COLLECT':
                if (directory/'STOP').exists(): status='STOPPED';break
                if state['active_seconds']>=3600*config['time_budget_hours']: status='TIME_BUDGET_COMPLETE';break
                index=state['sampler']['game_index']
                if index>=config['games']:
                    state['stage']='LABEL';event(True);break
                started=time.monotonic()
                if state['current_game'] is None:
                    game,opening_trace,family=go2_k20_opening(actor,generator,config,set(state['sampler']['opening_families'])|set(exclusions['families']))
                    state['sampler']['opening_families'].append(family)
                    profile=config['opponent_cycle'][index%len(config['opponent_cycle'])]
                    colour=1 if (index//len(config['opponent_cycle']))%2==0 else -1
                    state['current_game']={'state':go2_state_to_dict(game),'trace':opening_trace,'opening_family':family,'candidate_rows':[],
                        'profile':profile,'ahr_color':colour,'seed':config['seed']+index,'elapsed_seconds':0.0}
                    state['search']=None;live_tree=None;event(True)
                current=state['current_game'];game=go2_state_from_dict(current['state']);go2_verify_replay(game)
                elapsed_base=current['elapsed_seconds'];game_clock=time.monotonic()
                try:
                    if current['profile']!='self':
                        opponent=go2_bench_opponent_factory(project,current['profile'],current['seed'],9,config['komi'],
                            gtp_timeout_seconds=config['opponent_timeout_seconds'],
                            config_directory=project/'reports/ahr_v2'/(config['buffer_id']+'-collection')/'opponent_configs')
                        if go2_digest(opponent.pin)!=go2_digest(opponent_pins[current['profile']]): raise RuntimeError('Rollout opponent identity changed.')
                        opponent.replay(current['trace'])
                    unknown=None
                    while not game.terminal:
                        if game.move_number>=config['max_plies']: unknown='PLY_CUTOFF';break
                        if elapsed_base+time.monotonic()-game_clock>=config['game_time_seconds']: unknown='GAME_TIMEOUT';break
                        if (directory/'STOP').exists(): status='STOPPED';raise _Go2K20Pause()
                        actor_turn=current['profile']=='self' or game.to_play==current['ahr_color']
                        if actor_turn:
                            search_config=Go2SearchConfig(simulations=config['simulations'],batch_size=config['search_batch_size'],
                                seed=current['seed']*1000+game.move_number,wall_time_seconds=config['search_wall_seconds'],
                                root_noise_fraction=config['root_noise_fraction'],
                                reading_total_nodes=config['reading_nodes'],reading_wall_time_seconds=config['reading_wall_seconds'])
                            context={'purpose':config['purpose'],'guided_buffer':config['buffer_id'],'game_index':index,'ply':game.move_number}
                            if state['search'] is not None:
                                live_tree=go2_search_from_state_dict(state['search'],actor,game,search_config,context=context,project=project)
                            output=go2_search(actor,game,search_config,tree=live_tree,context=context,project=project,
                                on_batch=on_batch,evaluator=_go2_k20_fixture_evaluator if _fixture is not None else None)
                            action=int(output['action'])
                            successor=go2_step(game,action)
                            current['candidate_rows'].append({'state':go2_state_to_dict(game),'next_state':go2_state_to_dict(successor),
                                'action':action,'game_index':index,'ply':game.move_number,'colour':game.to_play,
                                'ahr_search_policy':copy.deepcopy(output['policy_target']),
                                'ahr_policy_valid':output['policy_target_valid'],'simulations':output['completed_simulations'],
                                'local_reading_facts':copy.deepcopy(output['reading_facts'])})
                            item={'kind':'play','color':game.to_play,'action':action,'ahr_move':True}
                            if opponent is not None: opponent.play(game.to_play,action)
                        else:
                            action=int(opponent.genmove(game.to_play));successor=go2_step(game,action)
                            item={'kind':'genmove','color':game.to_play,'action':action,'ahr_move':False}
                        current['trace'].append(item);game=successor;current['state']=go2_state_to_dict(game)
                        live_tree=None;state['search']=None;event()
                    if unknown is None and game.terminal:
                        result=go2_outcome(game);winner=result['winner'];score=result['score_black'];ownership=list(result['ownership_black'])
                    else: winner=None;score=None;ownership=None
                except (Go2OpponentError,TimeoutError,Go2IllegalMove) as exc:
                    if 'replay mismatch' in str(exc).lower() or 'regenerated a different' in str(exc): raise RuntimeError('Rollout opponent recovery mismatch; resume refused.') from exc
                    unknown=type(exc).__name__+': '+str(exc)[:200];winner=None;score=None;ownership=None
                finally:
                    if opponent is not None: opponent.close();opponent=None
                current['elapsed_seconds']=elapsed_base+time.monotonic()-game_clock
                candidates=current['candidate_rows']
                selection=go2_k20_select_rows(candidates,config['rows_per_game'],generator)
                ids=[item['index'] for item in selection]
                for item in sorted(selection,key=lambda x:x['index']):
                    row=copy.deepcopy(candidates[item['index']]);row.update(sampling_reason=item['sampling_reason'],priority_conditions=item['conditions']);row.update(winner=winner,final_score_black=score,
                        final_ownership_black=ownership,observed_outcome_valid=winner is not None,unknown_reason=unknown)
                    state['pending_rows'].append(row)
                state['replay'].append({'game_index':index,'profile':current['profile'],'ahr_color':current['ahr_color'],
                    'trace':copy.deepcopy(current['trace']),'opening_family':current['opening_family'],'final_state':go2_state_to_dict(game),'winner':winner,
                    'unknown_reason':unknown,'score_black':score,'plies':game.move_number,'selected_rows':len(ids)})
                state['sampler']['game_index']+=1;state['current_game']=None;state['search']=None;live_tree=None;game_clock=None
                state['active_seconds']+=time.monotonic()-started;event(True)
                _go2_k20_buffer_report(project,state,'RUNNING')
            if state['stage']=='LABEL':
                teacher=_Go2K20FixtureTeacher() if _fixture is not None else _go2_k20_teacher(project,config)
                def query(position,visits,allowed=None):
                    request={'state':go2_state_to_dict(position),'visits':visits,'allowed_actions':allowed,'teacher':teacher_pin}
                    key=go2_digest(request)
                    if key not in state['teacher_cache']:
                        answer=teacher.query(position,visits=visits,allowed_actions=allowed)
                        if not isinstance(answer,dict) or not {'policy','value'}<=set(answer): raise ValueError('Incomplete offline teacher answer.')
                        # Keep all query provenance and native estimates; never recast as proof.
                        state['teacher_cache'][key]=copy.deepcopy(answer);event(True)
                    return copy.deepcopy(state['teacher_cache'][key]),key
                while state['sampler']['query_index']<len(state['pending_rows']):
                    if (directory/'STOP').exists():status='STOPPED';break
                    if state['active_seconds']>=3600*config['time_budget_hours']:status='TIME_BUDGET_COMPLETE';break
                    started=time.monotonic();index=state['sampler']['query_index'];row=copy.deepcopy(state['pending_rows'][index])
                    position=go2_state_from_dict(row['state']);successor=go2_state_from_dict(row['next_state'])
                    try:
                        best,best_key=query(position,config['teacher_visits'])
                        forced,forced_key=query(position,config['teacher_visits'],[row['action']])
                        if successor.terminal: next_potential_black=0.0;next_key=None
                        else:
                            next_answer,next_key=query(successor,config['teacher_visits'])
                            next_potential_black=successor.to_play*float(next_answer['value'])
                        legal=np.asarray(go2_legal_mask(position),dtype=bool);policy=np.asarray(best['policy'],dtype=float)
                        if policy.shape!=(82,) or not np.isfinite(policy).all() or (policy<0).any(): raise ValueError('Teacher searched policy must be finite nonnegative 82 actions.')
                        policy=np.where(legal,policy,0.0)
                        if policy.sum()<=0 or best.get('policy_valid',True) is False: raise ValueError('Teacher policy target has no legal searched support.')
                        policy/=policy.sum();value=float(best['value']);selected_value=float(forced.get('move_values',{}).get(row['action'],forced['value']))
                        best_action_value=float(best.get('move_values',{}).get(best.get('best_action'),best['value']))
                        regret=go2_k20_opportunity_cost(best_action_value,selected_value)
                        credit=float(row['colour']*row['winner']-config['potential_strength']*value) if row['winner'] is not None else None
                        transition_reward=go2_k20_potential_reward(row['colour'],row['colour']*value,next_potential_black,
                            terminal=successor.terminal,winner=row['winner'] if successor.terminal else None,strength=config['potential_strength'])
                        row.update(teacher_policy=policy.tolist(),teacher_value=value,teacher_score=float(best.get('score',0.0)),
                            teacher_score_valid=bool(best.get('score_valid',False)),teacher_ownership=np.asarray(best.get('ownership',np.zeros(81)),dtype=float).reshape(81).tolist(),
                            teacher_ownership_valid=bool(best.get('ownership_valid',False)),selected_teacher_value=selected_value,
                            best_teacher_action_value=best_action_value,estimated_regret=regret,estimated_regret_units='win probability; 0.1 = 10 percentage points',
                            credit_return=credit,credit_valid=credit is not None,potential_transition_reward=transition_reward,
                            potential_black=row['colour']*value,next_potential_black=next_potential_black,
                            teacher_queries={'root':best_key,'selected':forced_key,'next':next_key},teacher_provenance=copy.deepcopy(best.get('provenance',{})))
                        state['labelled_rows'].append(row)
                    except (Go2OpponentError,TimeoutError) as exc:
                        state['label_failures'].append({'row_index':index,'status':'UNKNOWN','error':repr(exc),'state':row['state']})
                    state['sampler']['query_index']+=1;state['active_seconds']+=time.monotonic()-started;event(True)
                    _go2_k20_buffer_report(project,state,'RUNNING')
                if state['sampler']['query_index']==len(state['pending_rows']):
                    if not state['labelled_rows']: raise RuntimeError('No valid teacher labels; training refused.')
                    state['stage']='COMPLETE';event(True);status='COMPLETE'
        except _Go2K20Pause:
            status='STOPPED' if (directory/'STOP').exists() else 'PAUSED';save(True)
        except BaseException:
            save(True);_go2_k20_buffer_report(project,state,'ERROR');raise
        finally:
            if opponent is not None:opponent.close()
            if teacher is not None:teacher.close()
        _go2_k20_buffer_report(project,state,status)
        return state


### 3. Teacher anchors, calibrated Worker and separate economic credit

Both arms sample the same frozen labelled rollout rows and the same compatible teacher anchors. The economic multiplier affects **policy imitation only**. Value cross-entropy receives teacher win estimates and separately masked observed outcomes. Final score/ownership from completed AHR-rules games take priority over forecast auxiliaries; an incomplete game keeps only explicitly valid teacher estimates.

The economic credit head has its own output, so shaping never replaces the Worker win probability. Its MSE target is the actual signed terminal outcome minus the fixed teacher forecast at the sampled full-history state. The control keeps the same architecture and initialization but sets credit loss and regret weighting to zero. The head is an auxiliary learning signal, not an additional scoring rule used to choose moves.


In [ ]:
class Go2K20Network(Go2Network):
    def __init__(self,config=None):
        super().__init__(config)
        self.economic_credit=nn.Sequential(nn.Linear(2*self.config.width,self.config.width),nn.SiLU(),nn.Linear(self.config.width,1))
    def forward(self,features):
        output=super().forward(features)
        output['economic_credit']=2*torch.tanh(self.economic_credit(output['global_features'])).squeeze(1)
        return output

def go2_k20_row_batch(rows,indices,device='cpu',config=None):
    config=config or go2_k20_reward_config();chosen=[rows[int(i)] for i in indices]
    if not chosen: raise ValueError('At least one labelled rollout row is required.')
    states=[go2_state_from_dict(row['state']) for row in chosen]
    features=np.stack([go2_encode_state(state) for state in states]);legal=np.stack([go2_legal_mask(state) for state in states])
    outcomes=[];scores=[];score_valid=[];ownership=[];ownership_valid=[]
    for row in chosen:
        colour=row['colour'];known=row['winner'] is not None
        result=colour*row['winner'] if known else None
        outcomes.append([float(result==-1),float(result==0),float(result==1)] if known else [0.0,0.0,0.0])
        scores.append(colour*row['final_score_black']/(81+abs(config['komi'])) if known else row['teacher_score'])
        score_valid.append(known or row['teacher_score_valid'])
        ownership.append([colour*x for x in row['final_ownership_black']] if known else row['teacher_ownership'])
        ownership_valid.append(known or row['teacher_ownership_valid'])
    values={'features':features,'legal':legal,'policy':np.asarray([row['teacher_policy'] for row in chosen]),
        'value':np.asarray([row['teacher_value'] for row in chosen]),'score':np.asarray(scores),
        'score_valid':np.asarray(score_valid),'outcome':np.asarray(outcomes),
        'outcome_valid':np.asarray([row['winner'] is not None for row in chosen]),
        'ownership':np.asarray(ownership),'ownership_valid':np.asarray(ownership_valid),
        'row_weight':go2_k20_regret_weight([row['estimated_regret'] for row in chosen],config),
        'credit_return':np.asarray([row['credit_return'] if row['credit_valid'] else 0.0 for row in chosen]),
        'credit_valid':np.asarray([row['credit_valid'] for row in chosen]),
        'is_rollout':np.ones(len(chosen),dtype=bool)}
    boolean=('legal','score_valid','outcome_valid','ownership_valid','credit_valid','is_rollout')
    return {key:torch.as_tensor(value,dtype=torch.bool if key in boolean else torch.float32,device=device) for key,value in values.items()}

def _go2_k20_anchor_batch(dataset,indices,device):
    batch=go2_k18_sample_batch(dataset,indices,device=device,split='train')
    n=len(indices)
    batch['row_weight']=torch.as_tensor(batch.get('row_weight',torch.ones(n,device=device)),dtype=torch.float32,device=device)
    batch.update(credit_return=torch.zeros(n,device=device),credit_valid=torch.zeros(n,dtype=torch.bool,device=device),
                 is_rollout=torch.zeros(n,dtype=torch.bool,device=device))
    return batch

def go2_k20_merge_batches(anchor,rollout):
    keys=('features','legal','policy','value','score','score_valid','outcome','outcome_valid','ownership','ownership_valid',
          'row_weight','credit_return','credit_valid','is_rollout')
    # Older teacher ownership can be [N,9,9]; use a consistent 81-point layout.
    anchor=copy.copy(anchor);rollout=copy.copy(rollout)
    for batch in (anchor,rollout): batch['ownership']=batch['ownership'].reshape(len(batch['features']),81)
    return {key:torch.cat((anchor[key],rollout[key]),dim=0) for key in keys}

def go2_k20_loss(model,batch,config):
    go2_teacher_validate_batch(batch)
    n=len(batch['features']);weights=batch.get('row_weight',torch.ones(n,device=batch['features'].device)).float()
    if weights.shape!=(n,) or not torch.isfinite(weights).all() or (weights<=0).any(): raise ValueError('Positive finite row weights required.')
    output=model(batch['features']);policy_logits=(-output['scout_energy'].float()/model.config.scout_temperature).masked_fill(~batch['legal'].bool(),-1e9)
    policy_log=F.log_softmax(policy_logits,dim=1);value_log=F.log_softmax(-output['worker_energy'].float()/model.config.worker_temperature,dim=1)
    value=batch['value'].float().clamp(-1,1);soft=torch.stack(((1-value)/2,torch.zeros_like(value),(1+value)/2),dim=1)
    policy_rows=-(batch['policy'].float()*policy_log).sum(1)
    components={'policy':(weights*policy_rows).sum()/weights.sum(),
        'teacher_value':-(soft*value_log).sum(1).mean(),
        'game_outcome':go2_teacher_masked_mean(-(batch['outcome'].float()*value_log).sum(1),batch['outcome_valid']),
        'score':go2_teacher_masked_mean((output['score'].float()-batch['score'].float()).square(),batch['score_valid']),
        'ownership':go2_teacher_masked_mean((output['ownership'].float()-batch['ownership'].float().reshape(-1,81)).square().mean(1),batch['ownership_valid'])}
    credit=batch.get('credit_return',torch.zeros(n,device=weights.device)).float()
    credit_valid=batch.get('credit_valid',torch.zeros(n,dtype=torch.bool,device=weights.device)).bool()
    if credit.shape!=(n,) or credit_valid.shape!=(n,) or not torch.isfinite(credit).all() or (credit[credit_valid].abs()>2.00001).any():
        raise ValueError('Invalid separate economic return target.')
    components['economic_credit']=go2_teacher_masked_mean((output['economic_credit'].float()-credit).square(),credit_valid)
    loss=sum(config['loss_weights'][key]*components[key] for key in config['loss_weights'])+config['credit_loss_weight']*components['economic_credit']
    if not torch.isfinite(loss): raise RuntimeError('Non-finite guided training objective.')
    return {'loss':loss,'components':components,'output':output,'policy_log':policy_log,'value_log':value_log}

def _go2_k20_dataset(project,config):
    loader=globals().get('go2_k18_dataset')
    if loader is None: raise RuntimeError('Notebook 18 compatible dataset loader is required.')
    return loader(project,cache_id=config['cache_id'],verify=True)

def _go2_k20_rows(dataset,split):
    if 'splits' in dataset and split in dataset['splits']:
        entry=dataset['splits'][split]
        if isinstance(entry,dict) and 'features' in entry:return len(entry['features'])
    # The notebook 18 cache preserves the notebook 12/13 typed layout.
    return go2_teacher_split_count(dataset,split)

def _go2_k20_data_identity(dataset):
    if 'identity' in dataset: return copy.deepcopy(dataset['identity'])
    return go2_teacher_data_identity(dataset)

def _go2_k20_train_report(project,state,status):
    report={'run_id':state['config']['run_id'],'arm':state['config']['arm'],'status':status,
        'optimizer_step':state['global_step'],'optimizer_steps_requested':state['config']['optimizer_steps'],
        'buffer_id':state['config']['buffer_id'],'base_run_id':state['config']['base_run_id'],
        'metrics':copy.deepcopy(state['metrics']),'active_seconds':state['active_seconds'],
        'true_objective':'terminal win +1 / loss -1 / draw 0','economic_credit_is_separate_from_worker':True,
        'teacher_queries_during_play':False,'model_type':'Go2K20Network','updated_at_utc':go2_now()}
    directory=Path(project)/'reports/ahr_v2'/state['config']['run_id'];directory.mkdir(parents=True,exist_ok=True)
    go2_atomic_json(directory/'status.json',report);return report

def _go2_k20_fixture_dataset(project,config):
    position=go2_initial_state(9,7.5);legal=np.asarray(go2_legal_mask(position),dtype=bool);policy=legal.astype(float)/legal.sum()
    row={'features':go2_encode_state(position),'legal':legal,'policy':policy,'value':0.0,'score':0.0,
         'score_valid':True,'outcome':np.array([0.,0.,1.]),'outcome_valid':False,
         'ownership':np.zeros(81),'ownership_valid':False,'row_weight':1.0}
    return {'fixture':True,'identity':{'kind':'k20-synthetic-anchor-v1'},'arrays':row}

def _go2_k20_fixture_anchor(dataset,indices,device):
    n=len(indices);boolean=('legal','score_valid','outcome_valid','ownership_valid')
    batch={key:torch.as_tensor(np.repeat(np.asarray(value)[None],n,axis=0),
        dtype=torch.bool if key in boolean else torch.float32,device=device) for key,value in dataset['arrays'].items()}
    batch.update(credit_return=torch.zeros(n,device=device),credit_valid=torch.zeros(n,dtype=torch.bool,device=device),
                 is_rollout=torch.zeros(n,dtype=torch.bool,device=device))
    return batch

def go2_k20_guided_improve(project,config,max_events=None,on_checkpoint=None,*,_fixture=None):
    """One frozen teacher-guided iteration. Control/economics can share the exact same buffer."""
    project=go2_project(project);config=go2_k20_validate_config(config)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    torch.set_num_threads(config['cpu_threads'])
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
    if _fixture is not None and config['purpose']!='validation_fixture': raise ValueError('Disposable fixture purpose is required.')
    buffer=go2_k20_collect(project,config,max_events=max_events,on_checkpoint=on_checkpoint,_fixture=_fixture)
    if buffer['stage']!='COMPLETE': return {'status':'BUFFER_INCOMPLETE','buffer_state':buffer}
    dataset=_go2_k20_fixture_dataset(project,config) if _fixture is not None else _go2_k20_dataset(project,config)
    anchor_rows=4 if _fixture is not None else _go2_k20_rows(dataset,'train')
    dev_rows=4 if _fixture is not None else _go2_k20_rows(dataset,'dev')
    if anchor_rows<1 or dev_rows<1: raise RuntimeError('Both anchor train and development data are required.')
    identity={'schema':'ahr-go-v2-guided-training-v1','sources':go2_k20_sources(project),'config':config,
        'buffer_identity':copy.deepcopy(buffer['identity']),'buffer_rows_sha256':go2_digest(buffer['labelled_rows']),
        'anchor_data':_go2_k20_data_identity(dataset),'network':copy.deepcopy(buffer['model_metadata'].get('network',
            buffer['model_metadata'].get('config',{}).get('network',{}))),'model_type':'Go2K20Network'}
    if not identity['network']:
        # Verified notebook 19 loader metadata also exposes its exact network configuration.
        base,_=go2_k19_load_player(project,config['base_run_id'],which=config['base_which'],device='cpu')
        identity['network']=dataclasses.asdict(base.config)
    network=Go2NetConfig(**identity['network']);directory=project/'checkpoints'/config['run_id']
    go2_seed(config['seed']);torch.set_num_threads(config['cpu_threads'])
    if str(config['device']).startswith('cuda'):
        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
    model=Go2K20Network(network).to(config['device']);model.requires_grad_(True)
    loaded=model.load_state_dict(buffer['model'],strict=False)
    if loaded.unexpected_keys or any(not key.startswith('economic_credit.') for key in loaded.missing_keys):
        raise RuntimeError('Only the new credit head may be absent from the verified base weights.')
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:max(0.1,1-step/max(1,config['optimizer_steps'])))
    scaler=torch.amp.GradScaler('cuda',enabled=False)
    generator=torch.Generator(device='cpu').manual_seed(config['seed']+19)
    rows=buffer['labelled_rows'];rollout_count=max(1,config['batch_size']-int(config['batch_size']*config['anchor_fraction']))
    anchor_count=max(1,config['batch_size']-rollout_count)
    with go2_checkpoint_lock(directory):
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},
                'sampler':{'rows_seen':0,'rollout_rows_seen':0,'anchor_rows_seen':0},'global_step':0,'config':config,
                'metrics':{'last_loss':None,'recent_losses':[],'development_history':[],'last_development':None,
                    'best_development_loss':None,'observed_outcome_rows':0,'credit_rows':0},'active_seconds':0.0,
                'data_cursor':{'optimizer_step':0},'replay':{'buffer_identity':buffer['identity'],
                    'rows_sha256':identity['buffer_rows_sha256']},'distributed':{'kind':'single_writer_single_learner'},
                'development_indices':torch.randperm(dev_rows,generator=torch.Generator().manual_seed(config['seed']+55))[:min(dev_rows,config['development_rows'])],
                'model_metadata':copy.deepcopy(buffer['model_metadata'])}
        else:
            model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            go2_restore_rng(state['rng'],{'learner':generator});go2_invalidate_model_identity(model)
        last_saved=-1 if entry is None else state['global_step'];last_time=time.monotonic();last_update=state['global_step'];processed=0;pending_best=False
        def save(force=False):
            nonlocal entry,last_saved,last_time,last_update,pending_best
            if state['global_step']==last_saved:return
            if not force and time.monotonic()-last_time<config['checkpoint_seconds'] and state['global_step']-last_update<config['checkpoint_optimizer_steps']:return
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),scheduler=copy.deepcopy(scheduler.state_dict()),
                scaler=copy.deepcopy(scaler.state_dict()),rng=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'learner':generator}))
            state['data_cursor']['optimizer_step']=state['global_step']
            entry=go2_save_checkpoint(directory,state,best=pending_best,
                resume_cell='20: go2_k20_guided_improve(PROJECT,json.loads((PROJECT/"checkpoints"/run_id/"config.json").read_text()))')
            pending_best=False;last_saved=state['global_step'];last_time=time.monotonic();last_update=state['global_step']
            if on_checkpoint is not None:
                rng=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'learner':generator})
                try:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
                finally:go2_restore_rng(rng,{'learner':generator})
        def evaluate():
            nonlocal pending_best
            model.eval()
            if _fixture is not None:batch=_go2_k20_fixture_anchor(dataset,state['development_indices'],config['device'])
            else:
                batch=go2_k18_sample_batch(dataset,state['development_indices'],device=config['device'],split='dev')
                n=len(batch['features']);batch=copy.copy(batch)
                batch.update(row_weight=torch.ones(n,device=config['device']),credit_return=torch.zeros(n,device=config['device']),
                    credit_valid=torch.zeros(n,dtype=torch.bool,device=config['device']),is_rollout=torch.zeros(n,dtype=torch.bool,device=config['device']))
            with torch.no_grad():
                result=go2_k20_loss(model,batch,config)
                p=result['value_log'].exp();v=p[:,2]-p[:,0];known=batch['outcome_valid'].bool()
                audit={'optimizer_step':state['global_step'],'joint_loss':float(result['loss'].cpu()),
                    'teacher_value_mse':float((v-batch['value']).square().mean().cpu()),'observed_rows':int(known.sum()),
                    'observed_brier':float((p[known]-batch['outcome'][known]).square().sum(1).mean().cpu()) if known.any() else None,
                    'components':{key:float(value.cpu()) for key,value in result['components'].items()}}
            state['metrics']['last_development']=audit;state['metrics']['development_history'].append(audit)
            old=state['metrics']['best_development_loss']
            if old is None or audit['joint_loss']<old: state['metrics']['best_development_loss']=audit['joint_loss'];pending_best=True
        go2_atomic_json(directory/'config.json',config)
        if state['metrics']['last_development'] is None:evaluate()
        save(True);status='RUNNING';_go2_k20_train_report(project,state,status)
        try:
            while state['global_step']<config['optimizer_steps']:
                if (directory/'STOP').exists():status='STOPPED';break
                if state['active_seconds']>=3600*config['time_budget_hours']:status='TIME_BUDGET_COMPLETE';break
                if max_events is not None and processed>=max_events:status='PAUSED';break
                started=time.monotonic()
                anchor_ids=torch.randint(0,anchor_rows,(anchor_count,),generator=generator)
                rollout_ids=torch.randint(0,len(rows),(rollout_count,),generator=generator)
                anchor=_go2_k20_fixture_anchor(dataset,anchor_ids,config['device']) if _fixture is not None else _go2_k20_anchor_batch(dataset,anchor_ids,config['device'])
                rollout=go2_k20_row_batch(rows,rollout_ids,config['device'],config)
                batch=go2_k20_merge_batches(anchor,rollout)
                # D4 transforms all spatial targets consistently; scalar credit/regret remain unchanged.
                batch=go2_teacher_d4(batch,torch.randint(0,8,(len(batch['features']),),generator=generator))
                model.train();optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda' if str(config['device']).startswith('cuda') else 'cpu',dtype=torch.bfloat16,enabled=config['precision']=='bfloat16'):
                    result=go2_k20_loss(model,batch,config)
                result['loss'].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),config['gradient_clip'],error_if_nonfinite=True)
                optimizer.step();scheduler.step();go2_invalidate_model_identity(model)
                state['global_step']+=1;processed+=1;state['active_seconds']+=time.monotonic()-started
                state['sampler']['rows_seen']+=len(batch['features']);state['sampler']['rollout_rows_seen']+=rollout_count;state['sampler']['anchor_rows_seen']+=anchor_count
                metrics=state['metrics'];metrics['last_loss']=float(result['loss'].detach().cpu())
                record={'optimizer_step':state['global_step'],'loss':metrics['last_loss'],'row_weight_mean':float(batch['row_weight'].mean().cpu()),
                    'components':{key:float(value.detach().cpu()) for key,value in result['components'].items()},
                    'credit_rows':int(batch['credit_valid'].sum()),'observed_outcome_rows':int(batch['outcome_valid'].sum())}
                metrics['recent_losses']=(metrics['recent_losses']+[record])[-128:]
                metrics['observed_outcome_rows']+=record['observed_outcome_rows'];metrics['credit_rows']+=record['credit_rows']
                boundary=state['global_step']%config['evaluate_every_steps']==0 or state['global_step']==config['optimizer_steps']
                if boundary:evaluate()
                save(boundary)
                if boundary:_go2_k20_train_report(project,state,'RUNNING')
            else:status='COMPLETE'
        except BaseException as exc:
            # A partial optimizer update is discarded; resume only the last valid checkpoint.
            previous,_=go2_load_checkpoint(directory,identity)
            report=_go2_k20_train_report(project,previous,'ERROR');report.update(error=repr(exc),discarded_partial_update=True)
            go2_atomic_json(project/'reports/ahr_v2'/config['run_id']/'status.json',report);raise
        save(True);_go2_k20_train_report(project,state,status)
        return state

def go2_k20_load_player(project,run_id,which='best',device='cpu'):
    project=go2_project(project);directory=project/'checkpoints'/run_id
    state,entry=go2_load_checkpoint(directory,which=which)
    if state is None or state['identity'].get('schema')!='ahr-go-v2-guided-training-v1':raise RuntimeError('A valid notebook 20 trained checkpoint is required.')
    if go2_digest(state['identity']['sources'])!=go2_digest(go2_k20_sources(project)):raise RuntimeError('Notebook 20 source identity changed; loading refused.')
    model=Go2K20Network(Go2NetConfig(**state['identity']['network'])).to(device);model.load_state_dict(state['model'],strict=True)
    model.eval();model.requires_grad_(False)
    metadata={'run_id':run_id,'global_step':state['global_step'],'optimizer_step':state['global_step'],
        'checkpoint_file':entry['file'],'checkpoint_sha256':entry['sha256'],'model_identity':go2_model_identity(model),
        'model_type':'Go2K20Network','network':state['identity']['network'],'sources':state['identity']['sources'],
        'is_smoke':state['config']['purpose']=='validation_fixture','arm':state['config']['arm'],
        'data_identity':state['identity']['anchor_data'],'buffer_identity':state['identity']['buffer_identity'],
        'development_metrics':state['metrics']['last_development'],'true_objective':'terminal win',
        'economic_credit_used_to_choose_moves':False,'teacher_used_to_choose_moves':False}
    return model,metadata


### 4. Short checks and exact recovery

The validation below uses disposable synthetic fixtures, not main training games. It tests the alternating-player sign convention, path telescoping, draws, terminal boundaries, UNKNOWN rejection, legal passing, bounded weighting, pending search/reading recovery and exact optimizer recovery across fresh kernels. Real playing strength is assessed only by the separate fresh-game gates; this fixture makes no win-rate claim.


In [ ]:
def go2_k20_fixture_config(label,device='cpu'):
    config=go2_k20_reward_config(run_id='ahr-v2-k20-fixture-'+label,base_run_id='ahr-v2-k20-fixture-base',
        cache_id='k20-fixture',device=device,buffer_id='ahr-v2-k20-buffer-'+label)
    config.update(purpose='validation_fixture',games=2,rows_per_game=2,opponent_cycle=['self'],
        simulations=4,search_batch_size=2,search_wall_seconds=None,opening_plies=0,root_noise_fraction=0.0,reading_nodes=16,reading_wall_seconds=None,
        teacher_visits=2,batch_size=4,optimizer_steps=4,evaluate_every_steps=2,development_rows=4,
        checkpoint_optimizer_steps=1,precision='float32',cpu_threads=1)
    return config

def go2_k20_scientific_state(state,kind):
    if kind=='collection':
        keys=('model','rng','sampler','global_step','stage','replay','pending_rows','labelled_rows','label_failures',
              'teacher_cache','current_game','search','data_cursor','metrics')
    else:keys=('model','optimizer','scheduler','scaler','rng','sampler','global_step','metrics','development_indices','data_cursor')
    return {key:copy.deepcopy(state[key]) for key in keys}

def go2_k20_fixture_worker(project,label,kind,phase,device='cpu'):
    project=go2_project(project);directory=project/'reports/ahr_v2/fixtures/k20';directory.mkdir(parents=True,exist_ok=True)
    config=go2_k20_fixture_config(label,device);stem=label+'-'+kind+'-'+phase
    result_path=directory/(stem+'.pt')
    source="""from pathlib import Path
import nbformat,torch
PROJECT=Path(PROJECT_LITERAL)
for name in ['00_contracts_and_recovery.ipynb','15_teacher_evaluation_and_launch.ipynb']:
    for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),globals())
globals().update(go2_teacher_arena_namespace(PROJECT))
for prefix in ['17_','18_','19_','20_']:
    go2_import_exports(next((PROJECT/'experiments/ahr_v2').glob(prefix+'*.ipynb')),globals())
config=CONFIG_LITERAL
torch.set_num_threads(1)
if KIND_LITERAL=='collection':
    state=go2_k20_collect(PROJECT,config,max_events=2 if PHASE_LITERAL=='prefix' else None,_fixture={})
else:
    # Collection has its own complete immutable shared buffer before learner interruption.
    go2_k20_collect(PROJECT,config,_fixture={})
    state=go2_k20_guided_improve(PROJECT,config,max_events=2 if PHASE_LITERAL=='prefix' else None,_fixture={})
if 'buffer_state' in state:raise RuntimeError('Fixture did not reach optimizer stage.')
torch.save(go2_k20_scientific_state(state,KIND_LITERAL),RESULT_LITERAL)
print({'kind':KIND_LITERAL,'phase':PHASE_LITERAL,'step':state['global_step'],'rows':len(state.get('labelled_rows',[]))})
"""
    substitutions={'PROJECT_LITERAL':repr(str(project)),'CONFIG_LITERAL':repr(config),'KIND_LITERAL':repr(kind),
                   'PHASE_LITERAL':repr(phase),'RESULT_LITERAL':repr(str(result_path))}
    for token,value in substitutions.items():source=source.replace(token,value)
    path=directory/(stem+'.ipynb');worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(source)],
        metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(worker,path)
    log=directory/(stem+'.log')
    with log.open('wb') as stream:
        process=subprocess.run([str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
            '--ExecutePreprocessor.timeout=240',str(path)],cwd=project,stdout=stream,stderr=subprocess.STDOUT,timeout=300)
    if process.returncode or not result_path.is_file():raise RuntimeError('Fresh-kernel economic fixture failed: '+str(log)+' '+log.read_text()[-1800:])
    return torch.load(result_path,map_location='cpu',weights_only=False)

def go2_k20_validate(project=None,device='cpu'):
    project=go2_project(project);results={}
    for winner in (-1,0,1):
        for first in (-1,1):
            movers=[first,-first,first,-first];phi=[0.2,-0.4,0.7,-0.1,0.0]
            audit=go2_k20_potential_returns(movers,phi,winner=winner,strength=0.8)
            if not np.allclose(np.asarray(audit['returns'])+np.asarray(audit['fixed_state_baselines']),audit['unshaped_returns']):
                raise RuntimeError('Restoring the fixed baseline failed.')
    rejected=False
    try:go2_k20_potential_returns([1,-1],[0.1,0.2,0.0],winner=None)
    except ValueError:rejected=True
    if not rejected:raise RuntimeError('UNKNOWN was turned into an observed reward.')
    try:go2_k20_potential_reward(1,0.1,0.2,terminal=True,winner=1)
    except ValueError:results['terminal_zero_guard']=True
    else:raise RuntimeError('Nonzero terminal forecast changes the objective.')
    config=go2_k20_reward_config();weights=go2_k20_regret_weight([0,0.2,1.0],config)
    if not np.allclose(weights,[1,2,3]):raise RuntimeError('Regret weighting is not bounded as specified.')
    # A closed forecast path has zero net Black shaping even when intermediate signs alternate.
    rewards=[go2_k20_potential_reward(c,p,q,strength=1) for c,p,q in zip([1,-1,1],[0.2,0.8,-0.4],[0.8,-0.4,0.2])]
    if abs(sum(c*r for c,r in zip([1,-1,1],rewards)))>1e-12:raise RuntimeError('A forecast cycle earned net reward.')
    unique=uuid.uuid4().hex[:8]+'-'+device.replace(':','-')
    for kind in ('collection','learner'):
        full=go2_k20_fixture_worker(project,unique+'-'+kind+'-full',kind,'full',device)
        go2_k20_fixture_worker(project,unique+'-'+kind+'-resume',kind,'prefix',device)
        resumed=go2_k20_fixture_worker(project,unique+'-'+kind+'-resume',kind,'resume',device)
        if not go2_nested_equal(full,resumed):
            differing=[key for key in full if not go2_nested_equal(full[key],resumed[key])]
            raise RuntimeError('Fresh-kernel exact recovery differs for '+kind+': '+str(differing))
        results[kind+'_exact_recovery']=True
        results[kind+'_final_step']=full['global_step']
        if kind=='collection':
            if any(len(game['trace'])!=2 or game['winner']!=-1 for game in full['replay']):
                raise RuntimeError('Legal two-pass fixture did not terminate and score correctly.')
            unknown_row=copy.deepcopy(full['labelled_rows'][0])
            unknown_row.update(winner=None,final_score_black=None,final_ownership_black=None,credit_return=None,credit_valid=False)
            batch=go2_k20_row_batch([unknown_row],[0],device='cpu',config=config)
            if bool(batch['outcome_valid'][0]) or bool(batch['credit_valid'][0]) or float(batch['outcome'].sum())!=0:
                raise RuntimeError('UNKNOWN row acquired a terminal or credit target.')
            results['unknown_target_masks']=True
            results['legal_two_pass_scoring']=True
    # Both arms consume the identical labelled buffer; only bounded policy weights and credit loss differ.
    control=go2_k20_reward_config(arm='control');economics=go2_k20_reward_config(arm='economics')
    results.update(reward_telescoping=True,unknown_preserved=True,no_cycle_reward=True,regret_weights=weights.tolist(),
        control_credit_loss=control['credit_loss_weight'],economics_credit_loss=economics['credit_loss_weight'],
        fixture_only=True,device=device,source_identity=go2_k20_sources(project),validated_at_utc=go2_now())
    path=project/'reports/ahr_v2/fixtures/k20'/('validation-'+device.replace(':','-')+'.json');go2_atomic_json(path,results)
    return results


In [ ]:
EXECUTION_MODE=os.environ.get('AHR_K20_MODE','run')
if EXECUTION_MODE not in ('run','validate','status'):
    raise ValueError('Use AHR_K20_MODE run, validate, or status.')
if EXECUTION_MODE=='validate':
    VALIDATION=go2_k20_validate(PROJECT)
else:
    validation_path=PROJECT/'reports/ahr_v2/fixtures/k20/validation-cpu.json'
    VALIDATION=json.loads(validation_path.read_text()) if validation_path.exists() else go2_k20_validate(PROJECT)
    if VALIDATION.get('source_identity')!=go2_k20_sources(PROJECT):
        raise RuntimeError('Recovery receipt is stale. Execute with AHR_K20_MODE=validate before launching.')
display(pd.DataFrame([{'Check':key.replace('_',' '),'Result':value} for key,value in VALIDATION.items()
    if isinstance(value,(bool,int,float,str)) and key not in ('validated_at_utc',)]))


### 5. Credit accounting illustration

This deterministic four-transition example is a validation illustration, not a training result. Adding back each fixed forecast baseline recovers the true terminal return for that mover. The opportunity-cost multiplier is bounded and only affects teacher-policy imitation.


In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(9,3.2),'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
illustration=go2_k20_potential_returns([1,-1,1,-1],[0.2,-0.4,0.7,-0.1,0.0],winner=1,strength=1.0)
turn=np.arange(4);fig,axes=plt.subplots(1,2,figsize=(10,3.4))
axes[0].plot(turn,illustration['returns'],'o-',color='#286090',label='Separate credit return')
axes[0].plot(turn,illustration['unshaped_returns'],'s--',color='#444444',label='True win return')
axes[0].set(title='Credit accounting with a fixed teacher forecast\nValidation illustration; terminal winner stays fixed',xlabel='Mover turn',ylabel='Signed return')
axes[0].set_xticks(turn);axes[0].legend(fontsize=9);axes[0].grid(alpha=.2)
regret=np.linspace(0,1,101)
axes[1].plot(regret,go2_k20_regret_weight(regret,go2_k20_reward_config()),color='#286090',label='Economics')
axes[1].plot(regret,np.ones_like(regret),'--',color='#444444',label='Control')
axes[1].set(title='Bounded opportunity-cost weighting',xlabel='Estimated lost win probability',ylabel='Policy example weight',ylim=(0,3.2))
axes[1].legend();axes[1].grid(alpha=.2);fig.tight_layout();plt.show()


## Validation and the main run

The checks and illustration above concern disposable fixtures and reward accounting. They are not strength results. The cells below launch or resume the full registered workflow, show actual progress and provide a human-play panel. Failed promotion keeps the frozen incumbent.

### Pinned CUDA teacher dependencies

In [ ]:
from pathlib import Path
import os,json,hashlib,urllib.request,zipfile,subprocess,shutil

GO2_K20_GPU_URL='https://github.com/lightvector/KataGo/releases/download/v1.18.1/katago-v1.18.1-cuda13.2-cudnn9.24.0-linux-x64.zip'
GO2_K20_GPU_ARCHIVE_SHA='7294a8d0717726d7a8cc33015d2eca71496cfbef6c0fbb69e5f30d6711f9f689'

def go2_k20_gpu_assets(project,allow_download=False):
    project=Path(project).resolve();directory=project/'.venv/katago-k20-gpu';manifest_path=directory/'manifest.json'
    def sha(path):
        digest=hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda:stream.read(4*1024*1024),b''):digest.update(block)
        return digest.hexdigest()
    if not manifest_path.exists():
        if not allow_download:raise FileNotFoundError('GPU teacher assets not installed. Run the dependency setup cell.')
        directory.mkdir(parents=True,exist_ok=True);archive=directory/'katago-cuda13.2.zip'
        if not archive.exists():
            temporary=archive.with_suffix('.part')
            request=urllib.request.Request(GO2_K20_GPU_URL,headers={'User-Agent':'AHR-Go-notebook20'})
            with urllib.request.urlopen(request,timeout=60) as response,temporary.open('wb') as stream:
                shutil.copyfileobj(response,stream,4*1024*1024);stream.flush();os.fsync(stream.fileno())
            os.replace(temporary,archive)
        if sha(archive)!=GO2_K20_GPU_ARCHIVE_SHA:raise RuntimeError('GPU KataGo archive hash mismatch.')
        with zipfile.ZipFile(archive) as bundle:
            for item in bundle.infolist():
                if item.filename.startswith('/') or '..' in Path(item.filename).parts:raise RuntimeError('Unsafe third-party archive path.')
            bundle.extractall(directory/'release')
        appimages=[path for path in (directory/'release').rglob('*') if path.is_file() and (path.suffix=='.AppImage' or path.name=='katago')]
        if len(appimages)!=1:raise RuntimeError('Expected one official AppImage.')
        appimage=appimages[0];appimage.chmod(0o755)
        subprocess.run([str(appimage),'--appimage-extract'],cwd=directory,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,timeout=60)
        appdir=directory/'squashfs-root';executable=appdir/'AppRun';executable.chmod(0o755)
        for child in (appdir/'usr/bin').glob('*'):
            if child.is_file():child.chmod(0o755)
        environment_paths=[str(path) for path in sorted((project/'.venv/lib/python3.13/site-packages/nvidia').glob('*/lib'))]
        environment_paths+=[str(appdir/'usr/lib'),str(appdir/'usr/lib/x86_64-linux-gnu')]
        env=os.environ.copy();env['LD_LIBRARY_PATH']=':'.join(environment_paths+([env['LD_LIBRARY_PATH']] if env.get('LD_LIBRARY_PATH') else []))
        version=subprocess.run([str(executable),'version'],env=env,capture_output=True,text=True,check=True,timeout=30).stdout.strip()
        cpu_manifest=json.loads((project/'.venv/katago-benchmark17/manifest.json').read_text())
        model=Path(cpu_manifest['model'])
        if not model.is_absolute():model=project/model
        candidates=list(appdir.rglob('katago'));binaries=[path for path in candidates if path.is_file() and os.access(path,os.X_OK)]
        runtime_files={str(path):sha(path) for path in sorted(appdir.rglob('*.so*')) if path.is_file()}
        for folder in environment_paths:
            for pattern in ['libcudart.so.*','libcublas.so.*','libcublasLt.so.*','libcudnn*.so.*']:
                for path in Path(folder).glob(pattern):
                    if path.is_file():runtime_files[str(path)]=sha(path)
        manifest={'schema':'ahr-k20-offline-gpu-teacher-v1','release':'v1.18.1','archive':str(archive),'archive_sha256':GO2_K20_GPU_ARCHIVE_SHA,
                  'source_url':GO2_K20_GPU_URL,'executable':str(executable),'executable_sha256':sha(executable),
                  'binary_pins':{str(path):sha(path) for path in binaries},'version':version,'backend':'CUDA13.2-cuDNN9.24',
                  'model':str(model),'model_sha256':sha(model),'library_paths':environment_paths,'runtime_sha256':runtime_files}
        temporary=manifest_path.with_suffix('.part');temporary.write_text(json.dumps(manifest,indent=2)+'\n');os.replace(temporary,manifest_path)
    manifest=json.loads(manifest_path.read_text())
    if manifest.get('schema')!='ahr-k20-offline-gpu-teacher-v1':raise RuntimeError('Invalid GPU asset manifest.')
    for key in ['executable','model','archive']:
        if not Path(manifest[key]).is_file() or sha(manifest[key])!=manifest[key+'_sha256']:raise RuntimeError('GPU asset changed: '+key)
    for path,digest in {**manifest['binary_pins'],**manifest['runtime_sha256']}.items():
        if not Path(path).is_file() or sha(path)!=digest:raise RuntimeError('GPU dependency changed: '+path)
    environment=os.environ.copy();environment['LD_LIBRARY_PATH']=':'.join(manifest['library_paths']+([environment['LD_LIBRARY_PATH']] if environment.get('LD_LIBRARY_PATH') else []))
    return {**manifest,'env':{'LD_LIBRARY_PATH':environment['LD_LIBRARY_PATH']}}


### Detached workflow, comparison and promotion control

In [ ]:
import contextlib, shutil, subprocess, shlex, socket, tempfile, uuid

def go2_k20_gpu_teacher(project,visits=128,timeout_seconds=180,seed=2026101120,gpu_device=0):
    assets=go2_k20_gpu_assets(project,allow_download=False)
    env=os.environ.copy();env.update(assets['env'])
    return Go2K18Teacher(project,visits=visits,timeout_seconds=timeout_seconds,seed=seed,
                         assets=assets,env=env,gpu_device=int(gpu_device))

def go2_k20_control_sources(project):
    root=Path(project)/GO2_ROOT;result={}
    for number in range(21):
        paths=list(root.glob(str(number).zfill(2)+'_*.ipynb'))
        if len(paths)!=1:raise RuntimeError('Exactly one canonical notebook required: '+str(number))
        result[str(paths[0].relative_to(project))]=go2_code_sha(paths[0])
    return result

def go2_k20_pipeline_config(pipeline_id='ahr-v2-katago-econ-20261011-v1'):
    if Path(pipeline_id).name!=pipeline_id or not pipeline_id.startswith('ahr-v2-'):raise ValueError('Local ahr-v2- ID required.')
    return {'schema':'ahr-v2-katago-economic-pipeline-v1','pipeline_id':pipeline_id,
        'purpose':'development','seed':2026101120,'cache_id':'go9x9-competitive-v2',
        'train_device':'cuda:1','teacher_gpu_device':0,'arena_device':'cuda:1',
        'supervised_pair_id':'ahr-k19-20261011','supervised_hours_per_model':3.0,
        'supervised_optimizer_steps':20000,'supervised_epochs':6,
        'guided_games':32,'guided_rows_per_game':16,'guided_teacher_visits':128,
        'guided_optimizer_steps':1000,'guided_batch_size':256,'guided_anchor_fraction':0.75,
        'guided_hours_per_stage':8.0,'screen_games':12,'economic_screen_games':40,'confirmation_games':100,
        'previous_checkpoint_games':4,'stronger_katago_games':20,
        'incumbent_run_id':'ahr-v2-teacher-20261008','incumbent_candidate_file':'step-000008016-c19b207e.pt',
        'incumbent_checkpoint_sha256':'e7d1126c4ca962adb6eebcf5754341c4e6f1f3eb8396549e0fb8ee11e47fc1aa',
        'previous_loss_best_sha256':'a956e93c86456fff77b7fbbdc2d56a61d1587b4559169ddfa39402122f2eccf5',
        'gnu_regression_games':20,'ahr_move_seconds':2.0,'max_simulations':512,
        'promotion_policy':{'min_kata_games':100,'min_kata_wins':55,'min_gnu_games':20,
                            'min_gnu_win_rate':0.70,'max_gnu_drop':0.10,'max_unknown':0,
                            'ahr_soft_move_seconds':2.0,'min_opening_plies':8},
        'checkpoint_seconds':120,'checkpoint_steps':100,'test_consumed':False}

def go2_k20_pipeline_bootstrap(project):
    prefixes=('00_','01_','02_','03_','05_','08_','12_','13_','14_','15_','16_','17_','18_','19_','20_')
    names=[]
    for prefix in prefixes:
        paths=list((Path(project)/GO2_ROOT).glob(prefix+'*.ipynb'))
        if len(paths)!=1:raise RuntimeError('Canonical dependency missing or ambiguous: '+prefix)
        names.append(paths[0].name)
    return "from pathlib import Path\nimport os,nbformat,json\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nPROJECT=Path("+repr(str(project))+ ")\nfor cell in nbformat.read(PROJECT/'experiments/ahr_v2'/"+repr(names[0])+",4).cells:\n    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(cell.source,globals())\nfor name in "+repr(tuple(names[1:]))+":\n    go2_import_exports(PROJECT/'experiments/ahr_v2'/name,globals())\n"
def go2_k20_pipeline_paths(project,pipeline_id):
    project=go2_project(project)
    return project/'checkpoints'/pipeline_id,project/'reports/ahr_v2'/pipeline_id

def go2_k20_pipeline_preflight(project,config):
    project=go2_project(project)
    if config['purpose']!='development' or config['test_consumed'] is not False:raise ValueError('Development-only pipeline required.')
    if not torch.cuda.is_available() or torch.cuda.device_count()<2:raise RuntimeError('Both project GPUs are required.')
    free=shutil.disk_usage(project).free
    if free<20*1024**3:raise RuntimeError('At least20GiB free required for new bank and retained checkpoints.')
    receipts={}
    for key,file in [('data','k18_validation/receipt.json'),('training','k19_training_validation.json')]:
        path=project/'reports/ahr_v2'/file
        if not path.is_file():raise RuntimeError('Validated recovery receipt missing: '+str(path))
        receipts[key]=json.loads(path.read_text())
    if receipts['data'].get('status')!='PASS' or not receipts['data'].get('uninterrupted_resume_rows_identical') or not receipts['data'].get('full_latest_two_and_best_valid') or receipts['data'].get('sources')!=go2_k18_sources(project):raise RuntimeError('Notebook18 data recovery receipt is stale or failed.')
    if not receipts['training'].get('all_passed') or not all(any(f['device']==d and f.get('exact_scientific_state') for f in receipts['training'].get('fixtures',[])) for d in ('cpu',config['train_device'])):raise RuntimeError('Notebook19 CPU/GPU recovery validation failed.')
    if receipts['training'].get('source_identity')!=go2_k19_sources(project):raise RuntimeError('Stale notebook19 validation receipt.')
    for device in ('cpu','cuda-1'):
        path=project/'reports/ahr_v2/fixtures/k20'/('validation-'+device+'.json')
        if not path.is_file():raise RuntimeError('Notebook20 recovery receipt missing: '+device)
        receipt=json.loads(path.read_text())
        if receipt.get('source_identity')!=go2_k20_sources(project) or not receipt.get('collection_exact_recovery') or not receipt.get('learner_exact_recovery'):
            raise RuntimeError('Notebook20 exact recovery receipt is stale or failed: '+device)
        receipts[device]=receipt
    native_path=project/'reports/ahr_v2/fixtures/k20/stateless-teacher-validation.json'
    if not native_path.is_file():raise RuntimeError('Actual stateless GPU teacher validation receipt missing.')
    native=json.loads(native_path.read_text())
    if native.get('status')!='PASS' or not native.get('actual_gpu_predictions_exact') or not native.get('query_order_independent') or native.get('gpu_device')!=config['teacher_gpu_device'] or native.get('visits_per_query')!=config['guided_teacher_visits'] or go2_digest(native.get('protocol'))!=go2_digest(go2_k20_stateless_teacher_identity(project)):
        raise RuntimeError('Native teacher recovery/determinism receipt is stale or differs from the registered GPU/query budget.')
    receipts['native_teacher']=native
    control_path=project/'reports/ahr_v2/k20_pipeline_controller_validation.json'
    if not control_path.is_file():raise RuntimeError('Outer-controller exact recovery/tmux receipt missing.')
    controller=json.loads(control_path.read_text())
    if not controller.get('all_passed') or controller.get('sources')!=go2_k20_control_sources(project) or not controller.get('fresh_kernel_exact_stage_rng_resume') or not controller.get('detached_tmux_smoke'):
        raise RuntimeError('Outer-controller recovery receipt is stale or failed.')
    receipts['controller']=controller
    promotion_path=project/'reports/ahr_v2/k20_promotion_play_validation/receipt.json'
    if not promotion_path.is_file():raise RuntimeError('Promotion/player validation receipt missing.')
    promotion=json.loads(promotion_path.read_text())
    if promotion.get('status')!='PASS' or promotion.get('source_identity')!=go2_k20_sources(project) or not promotion.get('zero_wins_rejected') or not promotion.get('unknown_rejected') or not promotion.get('gnu_regression_rejected') or not promotion.get('human_history_exact_restore') or not promotion.get('pointer_and_snapshot_corruption_rejected'):
        raise RuntimeError('Promotion and human-player validation receipt is stale or failed.')
    receipts['promotion_player']=promotion
    assets=go2_k20_gpu_assets(project,allow_download=False)
    return {'sources':go2_k20_control_sources(project),'free_bytes':free,'receipts':receipts,
        'gpu_teacher_assets':{k:v for k,v in assets.items() if k!='env'},
        'storage':'persistent boot disk; instance-deletion policy unverified; off-instance backup deferred by user',
        'torch':torch.__version__,'gpus':[torch.cuda.get_device_name(i) for i in range(2)]}

def go2_k20_arena_namespace(project,config):
    ns=go2_k18_namespace(project)
    for prefix in ('19_','20_'):
        path=next((Path(project)/GO2_ROOT).glob(prefix+'*.ipynb'));go2_import_exports(path,ns)
    ns['go2_bench_sources']=ns['go2_k20_control_sources']
    old_exclusions=ns['go2_bench_prepared_exclusions']
    def exclusions(project,metadata):
        project=Path(project);old_meta=copy.deepcopy(metadata)
        old_meta['config']={'cache_id':'go9x9-teacher-v1'}
        result=old_exclusions(project,old_meta)
        manifest=go2_k18_dataset(project,config['cache_id'],verify=False)['manifest']
        result['train'].update(manifest['families']['train']);result['dev'].update(manifest['families']['dev'])
        result['cache_identity']={'previous':result['cache_identity'],'new':manifest['identity']}
        # Exclude only the current plan from its own freshness inventory. Earlier screens remain consumed.
        for path in sorted((project/'reports/ahr_v2/benchmarks').glob('*/plan.json')):
            if path.parent.name==config.get('_active_plan_id'):continue
            record=json.loads(path.read_text());families={g['family_sha'] for g in record.get('games',[]) if g.get('opening_actions')}
            if families:
                result['consumed'].update(families);result['prior_reviews'].append({'file':str(path.relative_to(project)),'sha256':go2_file_sha(path),'families':len(families)})
        buffer_id=config['pipeline_id']+'-buffer'
        directory=project/'checkpoints'/(buffer_id+'-collection')
        if (directory/'manifest.json').is_file():
            saved,_=go2_load_checkpoint(directory)
            families=set(saved['sampler'].get('opening_families',[]));result['train'].update(families)
            # Stable family list, never a rotating checkpoint hash, defines exclusion identity.
            result['cache_identity']['guided_training_families_sha']=go2_digest(sorted(families))
        return result
    ns['go2_bench_prepared_exclusions']=exclusions
    return ns

def go2_k20_arena_config(config,label,profiles):
    value=go2_bench_default_config()
    value.update(run_id=config['pipeline_id']+'-'+label,seed=config['seed']+sum(map(ord,label)),
        device=config['arena_device'],profiles=copy.deepcopy(profiles),empty_board_baseline=False,
        ahr_move_seconds=config['ahr_move_seconds'],max_simulations=config['max_simulations'],
        gtp_timeout_seconds=90.,game_time_seconds=1800.,speed_devices=[config['arena_device']],
        speed_positions=6,speed_move_seconds=[config['ahr_move_seconds']])
    return value

def go2_k20_arena_plan(project,config,label,profiles):
    active=copy.deepcopy(config);active['_active_plan_id']=config['pipeline_id']+'-'+label
    ns=go2_k20_arena_namespace(project,active)
    champion,metadata=go2_k20_frozen_incumbent(project,config,config['arena_device'])
    cfg=go2_k20_arena_config(config,label,profiles)
    return ns['go2_bench_plan'](project,cfg,champion,metadata),cfg

def go2_k20_arena_candidate(project,config,label,plan,arena_config,model,metadata):
    go2_k20_pipeline_check_stop(project,config)
    ns=go2_k20_arena_namespace(project,config);cfg=copy.deepcopy(arena_config)
    cfg['run_id']=config['pipeline_id']+'-'+label
    factory=lambda profile,seed,size,komi:ns['go2_bench_opponent_factory'](project,profile,seed,size,komi,
        gtp_timeout_seconds=cfg['gtp_timeout_seconds'],config_directory=Path(project)/'reports/ahr_v2'/config['pipeline_id']/'opponent_configs')
    report=ns['go2_bench_arena'](project,cfg['run_id'],cfg,plan,model,metadata,factory)
    if report['status']!='COMPLETE':
        if go2_k20_pipeline_stop_requested(project,config):raise _Go2K20PipelinePause('Arena paused at a valid checkpoint')
        raise RuntimeError('Arena did not complete; resume required: '+cfg['run_id'])
    return report

def go2_k20_profile_summary(report,profile):
    rows=[g for g in report['games'] if g.get('profile',g.get('opponent_profile',g.get('planned',{}).get('opponent_profile')))==profile]
    if not rows:raise RuntimeError('Completed arena has no games for required profile: '+profile)
    return _go2_bench_summary(rows)
def go2_k20_pipeline_stages(config):
    return ['diagnosis','prepare_competitive_data','previous_checkpoint_screen','train_control','train_challenger',
            'supervised_screen','collect_guided_buffer','guided_control','guided_economics',
            'economic_screen','fresh_confirmation','promotion_and_report','stronger_katago_gates']

class _Go2K20PipelinePause(Exception):
    pass

def go2_k20_frozen_incumbent(project,config,device='cpu'):
    model,metadata=go2_load_selected_teacher_player(project,run_id=config['incumbent_run_id'],
        candidate_file=config['incumbent_candidate_file'],device=device)
    if metadata['checkpoint_sha256']!=config['incumbent_checkpoint_sha256']:raise RuntimeError('Registered incumbent checkpoint changed')
    return model,metadata

def go2_k20_development_loss(metadata):
    audit=metadata.get('development') or metadata.get('development_metrics') or {}
    result=audit.get('joint_loss')
    return float(result) if result is not None and math.isfinite(float(result)) else None

def go2_k20_trained_k19_player(project,run_id,device):
    model,meta=go2_k19_load_player(project,run_id,which='best',device=device);which='best'
    if meta['optimizer_step']<=0:model,meta=go2_k19_load_player(project,run_id,which='latest',device=device);which='latest'
    if meta.get('is_smoke') or meta['optimizer_step']<=0:raise RuntimeError('No trained production candidate exists: '+run_id)
    return model,meta,which

def go2_k20_profile_subplan(plan,profile):
    result=copy.deepcopy(plan)
    result['games']=[g for g in plan['games'] if g.get('opponent_profile',g.get('profile'))==profile]
    if not result['games']:raise RuntimeError('Profile absent from registered common plan: '+profile)
    result['identity']['profile_derivation']={'source_plan_sha256':go2_digest(_go2_bench_plan_pin(plan)),'profile':profile}
    families={g['family_sha'] for g in result['games'] if g.get('opening_actions')}
    result['coverage'].update(games=len(result['games']),balanced_pairs=len(result['games'])//2,
        fresh_opening_families=len(families),empty_board_pairs=sum(not g.get('opening_actions') for g in result['games'])//2)
    return result

def go2_k20_pipeline_prior_families(project,config,exclude_plan_id=None):
    families=set()
    for path in sorted((Path(project)/'reports/ahr_v2/benchmarks').glob(config['pipeline_id']+'*/plan.json')):
        if path.parent.name==exclude_plan_id:continue
        record=json.loads(path.read_text());families.update(g['family_sha'] for g in record.get('games',[]) if g.get('opening_actions'))
    path=Path(project)/'checkpoints'/(config['pipeline_id']+'-buffer-collection')
    if (path/'manifest.json').exists():
        state,_=go2_load_checkpoint(path);families.update(state['sampler'].get('opening_families',[]))
    return sorted(families)

def go2_k20_pipeline_evidence(config,confirmation):
    root='reports/ahr_v2/benchmarks/'+config['pipeline_id']
    return {'katago_confirmation':root+'-confirmation-candidate-katago/report.json',
        'gnu_candidate':root+'-confirmation-candidate-gnu/report.json','gnu_champion':root+'-confirmation-incumbent-gnu/report.json',
        'training_cache_id':config['cache_id'],'additional_excluded_families':confirmation['additional_excluded_families']}

def go2_k20_matched_comparison(control,economics):
    def keys(report):return {(g['profile'],g['opening_family'],int(g['ahr_color'])):g for g in report['games']}
    old=keys(control);new=keys(economics)
    if len(old)!=len(control['games']) or len(new)!=len(economics['games']) or set(old)!=set(new):raise RuntimeError('Economics ablation requires unique matched opening/color games')
    known=[(old[k],new[k]) for k in old if old[k]['outcome']!='UNKNOWN' and new[k]['outcome']!='UNKNOWN']
    win_delta=sum(b['outcome']=='WIN' for a,b in known)-sum(a['outcome']=='WIN' for a,b in known)
    margins=[float(b['score_for_ahr'])-float(a['score_for_ahr']) for a,b in known if a.get('score_for_ahr') is not None and b.get('score_for_ahr') is not None]
    return {'matched_games':len(old),'paired_complete_games':len(known),'pairs_excluded_for_unknown':len(old)-len(known),
        'economics_minus_control_wins':win_delta,'mean_economics_minus_control_margin':float(np.mean(margins)) if margins else None,
        'margin_pairs':len(margins),'claim':'Descriptive development differences on shared openings; no confirmed improvement or Elo claim.'}

def go2_k20_pipeline_stop_paths(project,config):
    root=Path(project)/'checkpoints';ids={config['pipeline_id'],config['supervised_pair_id']+'-control',
        config['supervised_pair_id']+'-challenger',config['pipeline_id']+'-buffer-collection',
        config['pipeline_id']+'-guided-control',config['pipeline_id']+'-guided-economics'}
    ids.update(path.name for path in root.glob(config['pipeline_id']+'-*') if path.is_dir())
    return [root/name/'STOP' for name in sorted(ids)]

def go2_k20_pipeline_request_stop(project,config):
    paths=go2_k20_pipeline_stop_paths(project,config)
    for path in paths:go2_atomic_bytes(path,(go2_now()+'\n').encode())
    return {'status':'STOP_REQUESTED','pipeline_id':config['pipeline_id'],'markers':[str(p.relative_to(project)) for p in paths],
        'behavior':'Current complete update/batch or producer checkpoint is retained; subsequent stages do not start.'}

def go2_k20_pipeline_stop_requested(project,config):
    return any(path.exists() for path in go2_k20_pipeline_stop_paths(project,config))

def go2_k20_pipeline_check_stop(project,config):
    if go2_k20_pipeline_stop_requested(project,config):raise _Go2K20PipelinePause('Stop requested at a valid substage checkpoint')

def go2_k20_pipeline_stage(project,config,name,results):
    project=go2_project(project);runid=config['pipeline_id'];device=config['train_device']
    go2_k20_pipeline_check_stop(project,config)
    pair=go2_k19_pair_configs(config['supervised_pair_id'],config['cache_id'],device)
    for item in pair:item.update(time_budget_hours=config['supervised_hours_per_model'],
        max_optimizer_steps=config['supervised_optimizer_steps'],max_epochs=config['supervised_epochs'])
    if name=='diagnosis':
        model,metadata=go2_k20_frozen_incumbent(project,config,config['arena_device'])
        teacher=go2_k20_gpu_teacher(project,visits=config['guided_teacher_visits'],gpu_device=config['teacher_gpu_device'])
        try:report=go2_k18_diagnose(project,model=model,teacher=teacher,max_positions=8,simulations=128)
        finally:teacher.close()
        return {'report':str(project/'reports/ahr_v2/k18_diagnosis/report.json'),'scope':'consumed development positions; estimated teacher regret',
                'diagnostic_positions':len(report['diagnostic_positions'])}
    if name=='prepare_competitive_data':
        value=go2_k18_prepare(project,go2_k18_config({'cache_id':config['cache_id']}),
            on_commit=lambda _metrics:go2_k20_pipeline_check_stop(project,config))
        dataset=go2_k18_dataset(project,config['cache_id'],verify=True)
        return {'identity':dataset['identity'],'train_rows':dataset['train']['count'],'dev_rows':dataset['dev']['count']}
    if name in ('train_control','train_challenger'):
        variant=name.removeprefix('train_');item=next(c for c in pair if c['variant']==variant)
        value=go2_k19_train(project,item)
        if value['status'] in ('STOPPED','PAUSED'):raise _Go2K20PipelinePause('Supervised candidate paused')
        saved,entry=go2_load_checkpoint(project/'checkpoints'/item['run_id'])
        if saved['global_step']<=0:raise RuntimeError('Training produced no optimizer updates.')
        return {'run_id':item['run_id'],'global_step':saved['global_step'],'active_seconds':saved['active_seconds'],
            'last_development':saved['metrics']['last_development'],'latest_checkpoint':entry,'stop_reason':value['status'],
            'planned_optimizer_steps':item['max_optimizer_steps'],'actual_rows_seen':saved['sampler']['rows_seen'],
            'selection':'training development loss, separate from playing strength'}
    if name=='previous_checkpoint_screen':
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['previous_checkpoint_games']}]
        plan,cfg=go2_k20_arena_plan(project,config,'previous-checkpoint-screen-plan',profiles);outputs={}
        incumbent,old_meta=go2_k20_frozen_incumbent(project,config,device)
        final,final_meta=go2_load_teacher_player(project,config['incumbent_run_id'],which='best',device=device)
        if final_meta['checkpoint_sha256']!=config['previous_loss_best_sha256']:raise RuntimeError('Registered step-11724 comparison checkpoint changed')
        for label,model,meta in [('selected-8016',incumbent,old_meta),('loss-best-11724',final,final_meta)]:
            report=go2_k20_arena_candidate(project,config,'previous-'+label,plan,cfg,model,meta)
            outputs[label]={'summary':go2_k20_profile_summary(report,'katago-v1'),'checkpoint':meta}
        return {'candidates':outputs,'common_plan_sha':go2_digest(_go2_bench_plan_pin(plan)),
            'selection_is_development_only':True,'warm_start_remains_predeclared_step11724':True}
    if name=='supervised_screen':
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['screen_games']}]
        plan,cfg=go2_k20_arena_plan(project,config,'supervised-screen-plan',profiles);outputs={}
        for variant in ('control','challenger'):
            candidate=results['train_'+variant]['run_id'];model,meta,which=go2_k20_trained_k19_player(project,candidate,device)
            report=go2_k20_arena_candidate(project,config,'supervised-'+variant,plan,cfg,model,meta)
            summary=go2_k20_profile_summary(report,'katago-v1')
            loss=go2_k20_development_loss(meta)
            outputs[variant]={'run_id':candidate,'summary':summary,'development_loss':loss,'checkpoint':meta,'which':which}
        # Small screening result chooses an offline-teaching base; it does not promote a champion.
        selected=max(outputs,key=lambda k:(outputs[k]['summary']['WIN'],-outputs[k]['summary']['UNKNOWN'],-(outputs[k]['development_loss'] if outputs[k]['development_loss'] is not None else float('inf'))))
        return {'candidates':outputs,'selected_variant':selected,'base_run_id':outputs[selected]['run_id'],'base_which':outputs[selected]['which'],
                'selection_is_development_only':True,'common_plan_sha':go2_digest(_go2_bench_plan_pin(plan))}
    if name in ('collect_guided_buffer','guided_control','guided_economics'):
        arm='economics' if name=='guided_economics' else 'control'
        item=go2_k20_reward_config(run_id=runid+'-guided-'+arm,base_run_id=results['supervised_screen']['base_run_id'],
            cache_id=config['cache_id'],device=device,arm=arm,buffer_id=runid+'-buffer')
        item.update(base_which=results['supervised_screen']['base_which'],
            exclude_benchmark_run_ids=['ahr-v2-benchmark-20261009-v1',runid+'-previous-checkpoint-screen-plan',runid+'-supervised-screen-plan'],
            games=config['guided_games'],rows_per_game=config['guided_rows_per_game'],
            teacher_visits=config['guided_teacher_visits'],teacher_backend='cuda',teacher_gpu_device=config['teacher_gpu_device'],
            optimizer_steps=config['guided_optimizer_steps'],batch_size=config['guided_batch_size'],anchor_fraction=config['guided_anchor_fraction'],
            evaluate_every_steps=250,development_rows=4096,time_budget_hours=config['guided_hours_per_stage'])
        if name=='collect_guided_buffer':
            value=go2_k20_collect(project,item)
            if value['stage']!='COMPLETE':
                if go2_k20_pipeline_stop_requested(project,config):raise _Go2K20PipelinePause('Guided collection paused')
                raise RuntimeError('Guided collection incomplete; resume without changing config.')
            if not value['labelled_rows']:raise RuntimeError('Guided buffer contains no valid teacher rows; inspect label_failures before creating a new registered buffer.')
            return {'buffer_id':item['buffer_id'],'games':len(value['replay']),'labelled_rows':len(value['labelled_rows']),
                    'unknown_games':sum(g['winner'] is None for g in value['replay']),'identity':value['identity']}
        value=go2_k20_guided_improve(project,item)
        saved,entry=go2_load_checkpoint(project/'checkpoints'/item['run_id'])
        if saved['global_step']!=item['optimizer_steps']:
            if go2_k20_pipeline_stop_requested(project,config):raise _Go2K20PipelinePause('Guided learner paused')
            raise RuntimeError('Matched guided arm update budget incomplete; resume required.')
        return {'run_id':item['run_id'],'optimizer_steps':saved['global_step'],'arm':arm,'latest_checkpoint':entry,
                'metrics':saved['metrics'],'same_frozen_buffer':True,'teacher_used_to_choose_moves':False}
    if name=='economic_screen':
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['economic_screen_games']}]
        plan,cfg=go2_k20_arena_plan(project,config,'economic-screen-plan',profiles);outputs={};reports={}
        base=results['supervised_screen']['base_run_id'];model,meta=go2_k19_load_player(project,base,which=results['supervised_screen']['base_which'],device=device)
        candidates=[('base',model,meta)]
        for arm in ('control','economics'):
            model,meta=go2_k20_load_player(project,results['guided_'+arm]['run_id'],which='latest',device=device)
            candidates.append((arm,model,meta))
        for key,model,meta in candidates:
            report=go2_k20_arena_candidate(project,config,'economic-'+key,plan,cfg,model,meta)
            reports[key]=report
            outputs[key]={'summary':go2_k20_profile_summary(report,'katago-v1'),'metadata':meta,'which':results['supervised_screen']['base_which'] if key=='base' else 'latest'}
        # Predeclared tie preference retains the teacher anchor; no economics benefit claimed from a tie.
        priority={'base':2,'control':1,'economics':0}
        selected=max(outputs,key=lambda k:(outputs[k]['summary']['WIN'],-outputs[k]['summary']['UNKNOWN'],priority[k]))
        return {'candidates':outputs,'selected':selected,'common_plan_sha':go2_digest(_go2_bench_plan_pin(plan)),
            'economics_helped_on_small_screen':outputs['economics']['summary']['WIN']>outputs['control']['summary']['WIN'],
            'economics_effect_is_unconfirmed':True,'matched_comparison':go2_k20_matched_comparison(reports['control'],reports['economics'])}
    if name=='fresh_confirmation':
        screen=results['economic_screen'];selected=screen['selected'];meta=screen['candidates'][selected]['metadata']
        if screen['candidates'][selected]['summary']['WIN']==0:
            return {'status':'NO_SCREEN_WINS','promoted':False,'reason':'No candidate won the KataGo1 development screen; incumbent retained.',
                    'screen':screen,'economics_strength_effect':'unconfirmed'}
        if selected=='base':model,metadata=go2_k19_load_player(project,meta['run_id'],which=screen['candidates'][selected]['which'],device=device)
        else:model,metadata=go2_k20_load_player(project,meta['run_id'],which='latest',device=device)
        if metadata['model_identity']!=meta['model_identity']:raise RuntimeError('Selected candidate checkpoint changed before confirmation.')
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['confirmation_games']},
                  {'id':'gnugo-l10','kind':'gnugo','level':10,'games':config['gnu_regression_games']}]
        plan,cfg=go2_k20_arena_plan(project,config,'confirmation-plan',profiles)
        kata_plan=go2_k20_profile_subplan(plan,'katago-v1');gnu_plan=go2_k20_profile_subplan(plan,'gnugo-l10')
        kata_cfg=copy.deepcopy(cfg);kata_cfg['profiles']=[profiles[0]]
        gnu_cfg=copy.deepcopy(cfg);gnu_cfg['profiles']=[profiles[1]]
        candidate_report=go2_k20_arena_candidate(project,config,'confirmation-candidate-katago',kata_plan,kata_cfg,model,metadata)
        gnu_candidate_report=go2_k20_arena_candidate(project,config,'confirmation-candidate-gnu',gnu_plan,gnu_cfg,model,metadata)
        champion,champ_meta=go2_k20_frozen_incumbent(project,config,device)
        # Candidate and incumbent receive identical GNU prefixes, colors and seeds.
        champion_report=go2_k20_arena_candidate(project,config,'confirmation-incumbent-gnu',gnu_plan,gnu_cfg,champion,champ_meta)
        ns=go2_k20_arena_namespace(project,config)
        speed=ns['go2_bench_speed'](project,config['pipeline_id']+'-confirmation-candidate',kata_cfg,model,metadata,kata_plan)
        return {'status':'COMPLETE','candidate_metadata':metadata,'candidate_report':candidate_report,
            'gnu_candidate_report':gnu_candidate_report,'incumbent_report':champion_report,'speed':speed,'plan':plan,
            'kata_summary':go2_k20_profile_summary(candidate_report,'katago-v1'),
            'gnu_summary':go2_k20_profile_summary(gnu_candidate_report,'gnugo-l10'),
            'incumbent_gnu_summary':go2_k20_profile_summary(champion_report,'gnugo-l10'),
            'candidate_which':screen['candidates'][selected]['which'],
            'additional_excluded_families':go2_k20_pipeline_prior_families(project,config,exclude_plan_id=cfg['run_id'])}
    if name=='promotion_and_report':
        confirmation=results['fresh_confirmation']
        if confirmation['status']!='COMPLETE':return {'promoted':False,'reason':confirmation['reason'],'economics_strength_effect':'unconfirmed'}
        evidence=go2_k20_pipeline_evidence(config,confirmation)
        meta=confirmation['candidate_metadata']
        if meta.get('model_type')=='Go2K20Network':model,metadata=go2_k20_load_player(project,meta['run_id'],which='latest',device=device)
        else:model,metadata=go2_k19_load_player(project,meta['run_id'],which=confirmation['candidate_which'],device=device)
        return go2_k20_promote(project,config['pipeline_id'],model,metadata,evidence,policy=config['promotion_policy'])
    if name=='stronger_katago_gates':
        promotion=results['promotion_and_report']
        if not promotion.get('promoted',False):return {'status':'SKIPPED','reason':'KataGo1 promotion gate was not passed; incumbent retained.'}
        confirmation=results['fresh_confirmation'];meta=confirmation['candidate_metadata']
        if meta.get('model_type')=='Go2K20Network':model,metadata=go2_k20_load_player(project,meta['run_id'],which='latest',device=device)
        else:model,metadata=go2_k19_load_player(project,meta['run_id'],which=confirmation['candidate_which'],device=device)
        if metadata['model_identity']!=meta['model_identity']:raise RuntimeError('Promoted model changed before stronger gates')
        profiles=[{'id':'katago-v4','kind':'katago','visits':4,'games':config['stronger_katago_games']},
                  {'id':'katago-v16','kind':'katago','visits':16,'games':config['stronger_katago_games']}]
        plan,cfg=go2_k20_arena_plan(project,config,'stronger-katago-plan',profiles)
        report=go2_k20_arena_candidate(project,config,'stronger-katago-candidate',plan,cfg,model,metadata)
        return {'status':'COMPLETE','summaries':{p['id']:go2_k20_profile_summary(report,p['id']) for p in profiles},
            'claim':'Stronger pinned visit-cap development games; beating KataGo1 does not imply beating higher caps.'}
    raise ValueError('Unknown registered stage: '+name)

def go2_k20_pipeline_run(project,config,max_stages=None,*,_fixture=False):
    project=go2_project(project);config=copy.deepcopy(config);directory,report_dir=go2_k20_pipeline_paths(project,config['pipeline_id'])
    if _fixture and config['purpose']!='validation_fixture':raise ValueError('Fixture hook requires explicit disposable purpose.')
    sources=go2_k20_control_sources(project)
    identity={'schema':config['schema'],'sources':sources,'config':config}
    with go2_checkpoint_lock(directory):
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            go2_seed(config['seed'])
            state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'global_step':0,'model':{},'optimizer':{},'scheduler':{},'scaler':{},
                'rng':go2_capture_rng([]),'sampler':{'stage_index':0},'metrics':{},'replay':{},'distributed':{'kind':'single_writer_stage_controller'},'results':{}}
            entry=go2_save_checkpoint(directory,state,resume_cell='20: Run All with identical PIPELINE_CONFIG; launches/resumes detached worker.')
        else:go2_restore_rng(state['rng'])
        stages=['fixture-a','fixture-b','fixture-c'] if _fixture else go2_k20_pipeline_stages(config)
        processed=0
        while state['sampler']['stage_index']<len(stages):
            if max_stages is not None and processed>=max_stages:break
            if (directory/'STOP').exists():break
            name=stages[state['sampler']['stage_index']]
            status={'status':'RUNNING','stage':name,'completed_stages':list(state['results']),
                'pipeline_id':config['pipeline_id'],'updated_at_utc':go2_now(),'checkpoint':entry}
            go2_atomic_json(report_dir/'status.json',status)
            print(go2_now(),name,flush=True)
            try:
                result={'value':float(torch.rand(())),'step':state['global_step']} if _fixture else go2_k20_pipeline_stage(project,config,name,state['results'])
            except _Go2K20PipelinePause:
                break
            except BaseException as error:
                status.update(status='ERROR',error=repr(error));go2_atomic_json(report_dir/'status.json',status);raise
            state['results'][name]=result;state['sampler']['stage_index']+=1;state['global_step']+=1;processed+=1
            state['metrics']={'completed_stages':state['sampler']['stage_index'],'total_stages':len(stages)};state['rng']=go2_capture_rng([])
            entry=go2_save_checkpoint(directory,state,best=True,resume_cell='20: Run All with identical PIPELINE_CONFIG; exact stage/substage resume.')
            go2_atomic_json(report_dir/'results.json',{'pipeline_id':config['pipeline_id'],'results':state['results'],'checkpoint':entry,'sources':sources})
        complete=state['sampler']['stage_index']==len(stages)
        go2_atomic_json(report_dir/'status.json',{'pipeline_id':config['pipeline_id'],'status':'COMPLETE' if complete else 'PAUSED',
            'stage':None if complete else stages[state['sampler']['stage_index']],'completed_stages':list(state['results']),
            'checkpoint':entry,'updated_at_utc':go2_now()})
        return state

def go2_k20_pipeline_status(project,pipeline_id='ahr-v2-katago-econ-20261011-v1'):
    _,report_dir=go2_k20_pipeline_paths(project,pipeline_id)
    path=report_dir/'status.json'
    return json.loads(path.read_text()) if path.exists() else {'status':'NOT_STARTED','pipeline_id':pipeline_id}

def go2_k20_pipeline_launch(project,config,session='ahr-v2-katago-econ20',resume_stopped=False):
    project=go2_project(project);directory,report_dir=go2_k20_pipeline_paths(project,config['pipeline_id'])
    if not shutil.which('tmux'):raise RuntimeError('tmux is required.')
    if Path(session).name!=session or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in session):raise ValueError('Simple tmux session name required.')
    prior=report_dir/'pipeline_config.json';launch_path=report_dir/'launch.json'
    if prior.exists() and json.loads(prior.read_text())!=config:raise RuntimeError('Existing pipeline config differs; exact resume refused.')
    exists=subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0
    if exists:
        if not launch_path.exists():raise RuntimeError('Unregistered tmux session collision: '+session)
        existing=json.loads(launch_path.read_text())
        if existing.get('pipeline_id')!=config['pipeline_id'] or not prior.exists():raise RuntimeError('tmux session belongs to another pipeline or has no verified config.')
        live_preflight=report_dir/'preflight.json'
        if not live_preflight.exists() or json.loads(live_preflight.read_text()).get('sources')!=go2_k20_control_sources(project):raise RuntimeError('Current definitions differ from the registered running pipeline.')
        return {'status':'RUNNING','pipeline_id':config['pipeline_id'],'session':session,'attach':'tmux attach -t '+session,'detach':'Ctrl-b then d'}
    expected_identity={'schema':config['schema'],'sources':go2_k20_control_sources(project),'config':copy.deepcopy(config)}
    if (directory/'manifest.json').exists() or list(directory.glob('*.pt')):go2_load_checkpoint(directory,identity=expected_identity)
    status=go2_k20_pipeline_status(project,config['pipeline_id'])
    if status['status']=='COMPLETE':return status
    if go2_k20_pipeline_stop_requested(project,config):
        if not resume_stopped:return {'status':'STOP_REQUESTED','pipeline_id':config['pipeline_id'],'resume':'Click Resume or call launch with resume_stopped=True to remove this pipeline’s own STOP markers.'}
        for marker in go2_k20_pipeline_stop_paths(project,config):
            if marker.exists():marker.unlink()
    preflight=go2_k20_pipeline_preflight(project,config)
    go2_atomic_json(prior,config);go2_atomic_json(report_dir/'preflight.json',preflight)
    worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(go2_k20_pipeline_bootstrap(project)),
        nbformat.v4.new_code_cell('CONFIG=json.loads((PROJECT/'+repr(str(prior.relative_to(project)))+').read_text())\nRESULT=go2_k20_pipeline_run(PROJECT,CONFIG)')],
        metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
    worker_path=report_dir/'detached_worker.ipynb';nbformat.write(worker,worker_path)
    command=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
        '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(worker_path)]
    shell=shlex.join(command)+' >> '+shlex.quote(str(report_dir/'worker.log'))+' 2>&1'
    launch={'status':'STARTED','session':session,'pipeline_id':config['pipeline_id'],'started_at_utc':go2_now(),
        'attach':'tmux attach -t '+session,'detach':'Ctrl-b then d','worker_notebook':str(worker_path),
        'resume':'Run All with identical configuration after disconnection; an intentional STOP requires explicit Resume.',
        'timing':'Six hours of supervised-training caps plus collection/ablation/evaluation; actual stop can be much earlier when registered updates finish.'}
    # Publish registration before tmux starts so a repeated click can verify session ownership.
    go2_atomic_json(launch_path,launch)
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),shell],check=True)
    return launch


In [ ]:
def go2_k20_pipeline_fixture_worker(project,config,steps,label,export_path):
    from nbclient import NotebookClient
    folder=Path(project)/'reports/ahr_v2/fixtures/k20-controller'/config['pipeline_id'];folder.mkdir(parents=True,exist_ok=True)
    path=folder/(label+'.ipynb');bootstrap=go2_k20_pipeline_bootstrap(project)
    canonical=Path(project)/GO2_ROOT/'20_economic_rewards_and_teacher_guided_improvement.ipynb'
    if Path(export_path).resolve()!=canonical.resolve():bootstrap+="\ngo2_import_exports(Path("+repr(str(export_path))+"),globals())\n"
    run="CONFIG=json.loads("+repr(json.dumps(config))+")\nRESULT=go2_k20_pipeline_run(PROJECT,CONFIG,max_stages="+str(int(steps))+",_fixture=True)\nprint({'stage_index':RESULT['sampler']['stage_index'],'step':RESULT['global_step']})"
    notebook=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Disposable CPU stage-controller recovery fixture; no Go games, teacher calls or model training.'),
        nbformat.v4.new_code_cell(bootstrap),nbformat.v4.new_code_cell(run)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
    nbformat.write(notebook,path);NotebookClient(notebook,timeout=90,kernel_name='ahr-go',resources={'metadata':{'path':str(project)}}).execute();nbformat.write(notebook,path)
    state,_=go2_load_checkpoint(Path(project)/'checkpoints'/config['pipeline_id']);return state

def go2_k20_pipeline_validate(project,export_path=None,check_tmux=True):
    project=go2_project(project);token=uuid.uuid4().hex[:10]
    canonical=project/GO2_ROOT/'20_economic_rewards_and_teacher_guided_improvement.ipynb'
    if export_path is None:
        present=any(c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[]) and 'def go2_k20_pipeline_validate' in c.source for c in nbformat.read(canonical,4).cells)
        export_path=canonical if present else project/'reports/ahr_v2/fixtures/k20_pipeline_control.ipynb'
    export_path=Path(export_path)
    config=go2_k20_pipeline_config('ahr-v2-controller-fixture-'+token+'-continuous')
    config.update(purpose='validation_fixture',_fixture_controller_export_sha=go2_code_sha(export_path))
    full=go2_k20_pipeline_fixture_worker(project,config,3,'continuous',export_path)
    resumed_config=copy.deepcopy(config);resumed_config['pipeline_id']='ahr-v2-controller-fixture-'+token+'-resumed'
    partial=go2_k20_pipeline_fixture_worker(project,resumed_config,1,'interrupted',export_path)
    resumed=go2_k20_pipeline_fixture_worker(project,resumed_config,2,'resumed',export_path)
    scientific=lambda state:{key:state[key] for key in ('global_step','rng','sampler','metrics','results','model','optimizer','scheduler','scaler')}
    if not go2_nested_equal(scientific(full),scientific(resumed)):raise AssertionError('Fresh-kernel stage/RNG recovery differs')
    if partial['sampler']['stage_index']!=1 or resumed['sampler']['stage_index']!=3:raise AssertionError('Interrupted controller cursor is incorrect')
    directory=project/'checkpoints'/resumed_config['pipeline_id'];wrong=copy.deepcopy(resumed['identity']);wrong['config']['seed']+=1
    try:go2_load_checkpoint(directory,identity=wrong)
    except RuntimeError:pass
    else:raise AssertionError('Changed controller configuration accepted')
    wrong=copy.deepcopy(resumed['identity']);wrong['sources'][next(iter(wrong['sources']))]='0'*64
    try:go2_load_checkpoint(directory,identity=wrong)
    except RuntimeError:pass
    else:raise AssertionError('Changed controller source accepted')
    with tempfile.TemporaryDirectory(prefix='k20-controller-corrupt-',dir=project/'checkpoints') as temporary:
        bad=Path(temporary);shutil.copytree(directory,bad,dirs_exist_ok=True);manifest=json.loads((bad/'manifest.json').read_text())
        payload=bad/manifest['latest'];raw=bytearray(payload.read_bytes());raw[len(raw)//2]^=1;payload.write_bytes(raw)
        try:go2_load_checkpoint(bad)
        except RuntimeError:pass
        else:raise AssertionError('Corrupt controller silently restarted')
    stop_config=copy.deepcopy(config);stop_config['pipeline_id']='ahr-v2-controller-fixture-'+token+'-stop';stop_config['supervised_pair_id']='ahr-k19-fixture-'+token
    go2_k20_pipeline_request_stop(project,stop_config)
    if not go2_k20_pipeline_stop_requested(project,stop_config):raise AssertionError('Stop request was not visible')
    paused=go2_k20_pipeline_run(project,stop_config,_fixture=True)
    if paused['sampler']['stage_index']!=0:raise AssertionError('Stopped controller advanced a stage')
    for marker in go2_k20_pipeline_stop_paths(project,stop_config):
        if marker.exists():marker.unlink()
    tmux_result=None
    if check_tmux:
        tmux_config=copy.deepcopy(config);tmux_config['pipeline_id']='ahr-v2-controller-fixture-'+token+'-tmux'
        folder=project/'reports/ahr_v2/fixtures/k20-controller'/tmux_config['pipeline_id'];folder.mkdir(parents=True,exist_ok=True)
        bootstrap=go2_k20_pipeline_bootstrap(project)
        if export_path.resolve()!=canonical.resolve():bootstrap+="\ngo2_import_exports(Path("+repr(str(export_path))+"),globals())\n"
        result_path=folder/'returned.json';worker_path=folder/'detached.ipynb'
        code="CONFIG=json.loads("+repr(json.dumps(tmux_config))+")\nRESULT=go2_k20_pipeline_run(PROJECT,CONFIG,_fixture=True)\ngo2_atomic_json(Path("+repr(str(result_path))+"),{'status':'RETURNED','stages':RESULT['sampler']['stage_index']})"
        worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(bootstrap),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
        nbformat.write(worker,worker_path);session='ahr-k20-controller-fixture-'+token
        command=shlex.join([str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
            '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=90',str(worker_path)])+' > '+shlex.quote(str(folder/'tmux.log'))+' 2>&1'
        subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True)
        deadline=time.monotonic()+45
        while not result_path.exists() and time.monotonic()<deadline:time.sleep(.5)
        if not result_path.exists():raise AssertionError('Detached fixture failed; inspect '+str(folder/'tmux.log'))
        tmux_result=json.loads(result_path.read_text())
        if tmux_result.get('status')!='RETURNED' or tmux_result.get('stages')!=3:raise AssertionError('Detached controller did not complete exactly three disposable stages')
    report={'schema':'ahr-k20-pipeline-controller-validation-v1','all_passed':True,'sources':go2_k20_control_sources(project),
        'controller_export_sha256':go2_code_sha(export_path),'fresh_kernel_exact_stage_rng_resume':True,
        'configuration_mismatch_refused':True,'source_mismatch_refused':True,'corruption_refused':True,
        'stop_preserves_stage_cursor':True,'detached_tmux_smoke':tmux_result,'production_started':False,'validated_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/k20_pipeline_controller_validation.json',report);return report

### Deterministic offline teacher queries

In [ ]:
GO2_K20_STATELESS_MODULE='experiments/ahr_v2/20_economic_rewards_and_teacher_guided_improvement.ipynb'
GO2_K20_STATELESS_SETTINGS={'forDeterministicTesting':True,'numAnalysisThreads':1,
    'numSearchThreadsPerAnalysisThread':1,'numEigenThreadsPerModel':1,'nnMaxBatchSize':1,
    'nnRandomize':False,'rootNoiseEnabled':False,'wideRootNoise':0,
    'rootNumSymmetriesToSample':1,'rootSymmetryPruning':False,'useEvalCache':False}

def go2_k20_stateless_teacher_identity(project):
    return {'schema':'ahr-k20-stateless-teacher-v1','processes_per_query':1,'requests_per_process':1,
        'stable_query_id':'1','search_seed_source':'KataGo v1.18.1 forDeterministicTesting fixed SeedRand',
        'query_order_independent':True,'settings':copy.deepcopy(GO2_K20_STATELESS_SETTINGS),
        'cuda_precision':{'FP16':False,'NHWC':False},
        'inherited_rules_history_reader_sha256':go2_code_sha(Path(project)/GO2_ROOT/'18_diagnosis_and_stronger_teacher_data.ipynb'),
        'definition_module':GO2_K20_STATELESS_MODULE,
        'definition_export_sha256':go2_code_sha(Path(project)/GO2_K20_STATELESS_MODULE),
        'comparison_scope':'Same pinned executable, model, rules, configuration, GPU and dependencies; no cross-hardware bit-exact claim'}

class Go2K20OneShotTeacher(Go2K18Teacher):
    """Inherit the verified 18 full-history reader; configure determinism before one process is spawned."""
    def __init__(self,project,visits=128,timeout_seconds=120,seed=2026101120,assets=None,env=None,gpu_device=0):
        self.project=Path(project)
        self.assets=copy.deepcopy(assets) if assets is not None else go2_bench_katago_assets(project,allow_download=False)
        # Retain the exact source reader's strict dependency checks on every query.
        if self.assets.get('model_sha256')!=GO2_TEACHER_HASH or go2_file_sha(self.assets['model'])!=GO2_TEACHER_HASH:
            raise RuntimeError('Stateless teacher model identity mismatch.')
        if self.assets.get('executable_sha256') and go2_file_sha(self.assets['executable'])!=self.assets['executable_sha256']:
            raise RuntimeError('Stateless teacher executable checksum mismatch.')
        for field in ['binary_pins','runtime_sha256']:
            for asset_path,sha in self.assets.get(field,{}).items():
                if Path(asset_path).is_absolute() and go2_file_sha(asset_path)!=sha:
                    raise RuntimeError('Stateless teacher dependency checksum mismatch: '+asset_path)
        self.environment=dict(os.environ);self.environment.update(env or {});self.gpu_device=int(gpu_device)
        self.visits=int(visits);self.timeout_seconds=float(timeout_seconds);self.sequence=0;self.buffer=bytearray();self._query_used=False
        if self.visits<2 or self.timeout_seconds<=0:raise ValueError('At least two teacher visits and a positive timeout are required.')
        settings={'numAnalysisThreads':1,'numSearchThreadsPerAnalysisThread':1,'numEigenThreadsPerModel':1,
            'nnMaxBatchSize':1,'nnCacheSizePowerOfTwo':12,'nnMutexPoolSizePowerOfTwo':8,
            'maxVisits':self.visits,'nnRandomize':False,'nnRandSeed':str(seed),
            'rootNumSymmetriesToSample':1,'rootSymmetryPruning':False,'rootNoiseEnabled':False,
            'chosenMoveTemperature':0,'chosenMoveTemperatureEarly':0,'reportAnalysisWinratesAs':'SIDETOMOVE',
            'logToStderr':False,'logAllRequests':False,'logAllResponses':False,'logSearchInfo':False,
            'wideRootNoise':0,'ignorePreRootHistory':False,'conservativePass':False,'preventCleanupPhase':True,
            'useEvalCache':False}
        settings.update(copy.deepcopy(GO2_K20_STATELESS_SETTINGS))
        if 'CUDA' in self.assets.get('backend',''):
            settings.update(cudaUseFP16=False,cudaUseNHWC=False,cudaDeviceToUse=int(gpu_device))
        text='\n'.join(key+' = '+(str(value).lower() if isinstance(value,bool) else str(value)) for key,value in settings.items())+'\n'
        directory=Path(project)/'reports/ahr_v2/k20_stateless_teacher_configs';directory.mkdir(parents=True,exist_ok=True)
        self.config_path=directory/('analysis-'+hashlib.sha256(text.encode()).hexdigest()[:20]+'.cfg')
        if self.config_path.exists() and self.config_path.read_text()!=text:raise RuntimeError('Deterministic teacher configuration collision.')
        if not self.config_path.exists():go2_atomic_bytes(self.config_path,text.encode())
        self.identity={'assets':copy.deepcopy(self.assets),'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),
            'config_sha256':go2_file_sha(self.config_path),'visits':int(visits),'gpu_device':int(gpu_device),
            'stateless_protocol':go2_k20_stateless_teacher_identity(project)}
        self.stderr_path=directory/('oneshot-'+str(os.getpid())+'-'+str(time.monotonic_ns())+'.log')
        self.stderr=self.stderr_path.open('xb')
        try:
            self.process=subprocess.Popen([self.assets['executable'],'analysis','-model',self.assets['model'],
                '-config',str(self.config_path),'-quit-without-waiting'],cwd=directory,
                stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=self.stderr,bufsize=0,env=self.environment)
            self.selector=selectors.DefaultSelector();self.selector.register(self.process.stdout,selectors.EVENT_READ)
        except BaseException:
            self.close();raise
    def query(self,state,visits=None,allowed_actions=None):
        if self._query_used:raise RuntimeError('A one-shot teacher process accepts exactly one analysis request.')
        self._query_used=True
        try:
            answer=super().query(state,visits=visits,allowed_actions=allowed_actions)
            if answer['query']['id']!='1':raise RuntimeError('Stateless teacher query ID drifted.')
            return answer
        finally:self.close()

class Go2K20StatelessTeacher:
    def __init__(self,project,visits=128,timeout_seconds=120,seed=2026101120,assets=None,env=None,gpu_device=0):
        self.project=Path(project);self.visits=int(visits);self.timeout_seconds=float(timeout_seconds)
        self.seed=int(seed);self.assets=copy.deepcopy(assets) if assets is not None else go2_bench_katago_assets(project,allow_download=False)
        self.environment=dict(os.environ);self.environment.update(env or {});self.gpu_device=int(gpu_device)
        self.identity={'assets':copy.deepcopy(self.assets),'stateless_protocol':go2_k20_stateless_teacher_identity(project),
            'visits':self.visits,'gpu_device':self.gpu_device};self._active=None
        if self.visits<2 or self.timeout_seconds<=0:raise ValueError('A bounded searched teacher query is required.')
    def query(self,state,visits=None,allowed_actions=None):
        if self._active is not None:raise RuntimeError('Only one query may own a stateless teacher actor.')
        started=time.monotonic();worker=None
        try:
            worker=Go2K20OneShotTeacher(self.project,visits=int(visits or self.visits),timeout_seconds=self.timeout_seconds,
                seed=self.seed,assets=self.assets,env=self.environment,gpu_device=self.gpu_device)
            self._active=worker
            answer=worker.query(state,visits=visits,allowed_actions=allowed_actions)
            answer['analysis_query_seconds']=answer['elapsed_seconds']
            answer['elapsed_seconds']=time.monotonic()-started
            answer['stateless_protocol']=copy.deepcopy(self.identity['stateless_protocol'])
            return answer
        finally:
            if worker is not None:worker.close()
            self._active=None
    def regret(self,state,chosen_action,visits=None):
        best=self.query(state,visits=visits);forced=self.query(state,visits=visits,allowed_actions=[int(chosen_action)])
        best_value=best['move_values'].get(best['best_action'],best['value'])
        chosen_value=forced['move_values'].get(int(chosen_action),forced['value'])
        return {'best':best,'chosen':forced,'estimated_winrate_regret':max(0.,(best_value-chosen_value)/2),
            'estimated_score_regret':best['score_points']-forced['score_points'],'certainty':'teacher_estimate'}
    def close(self):
        if self._active is not None:self._active.close();self._active=None
    def __enter__(self):return self
    def __exit__(self,*args):self.close()

def go2_k20_gpu_teacher(project,visits=128,timeout_seconds=120,seed=2026101120,gpu_device=0):
    assets=go2_k20_gpu_assets(project);environment=dict(os.environ);environment.update(assets.get('env',{}))
    if 'CUDA' not in assets.get('backend',''):raise RuntimeError('Explicit GPU teacher asset manifest required; no CPU fallback.')
    return Go2K20StatelessTeacher(project,visits=visits,timeout_seconds=timeout_seconds,seed=seed,
        assets=assets,env=environment,gpu_device=gpu_device)

def go2_k20_cpu_teacher(project,visits=128,timeout_seconds=120,seed=2026101120):
    assets=go2_bench_katago_assets(project,allow_download=False)
    return Go2K20StatelessTeacher(project,visits=visits,timeout_seconds=timeout_seconds,seed=seed,assets=assets)

def go2_k20_teacher_scientific(answer):
    keys=('policy','value','score','score_points','score_valid','ownership','ownership_valid','move_values',
          'move_scores','visits','best_action','query','response','query_sha256','response_sha256')
    # elapsed_seconds and hardware timing are not scientific prediction state.
    return {key:copy.deepcopy(answer[key]) for key in keys if key in answer}

def go2_k20_validate_stateless_teacher(project,visits=128,gpu_device=0):
    project=Path(project);states=[go2_initial_state(9,7.5)]
    position=states[0]
    for action in [40,10,30,50]:position=go2_step(position,action)
    states.append(position);answers={};durations=[]
    teacher=go2_k20_gpu_teacher(project,visits=visits,gpu_device=gpu_device)
    try:
        for index,state in enumerate(states):
            for kind in ('root','forced'):
                allowed=None if kind=='root' else [answers[(index,'root')]['best_action']]
                answer=teacher.query(state,visits=visits,allowed_actions=allowed)
                answers[(index,kind)]=answer;durations.append(answer['elapsed_seconds'])
        # Simulate a resumed label stream: new factory, reversed request order, fresh process for each query.
        teacher.close();teacher=go2_k20_gpu_teacher(project,visits=visits,gpu_device=gpu_device)
        for index,kind in [(1,'forced'),(0,'forced'),(1,'root'),(0,'root')]:
            allowed=None if kind=='root' else [answers[(index,'root')]['best_action']]
            repeated=teacher.query(states[index],visits=visits,allowed_actions=allowed);durations.append(repeated['elapsed_seconds'])
            if not go2_nested_equal(go2_k20_teacher_scientific(answers[(index,kind)]),go2_k20_teacher_scientific(repeated)):
                first=go2_k20_teacher_scientific(answers[(index,kind)]);second=go2_k20_teacher_scientific(repeated)
                keys=[key for key in first if not go2_nested_equal(first[key],second[key])]
                raise RuntimeError('Actual GPU teacher exact recovery failed: '+str((index,kind,keys)))
        # A real timeout must close its process and remain UNKNOWN; no fictitious label is returned.
        timeout_teacher=go2_k20_gpu_teacher(project,visits=visits,timeout_seconds=0.001,gpu_device=gpu_device)
        try:
            try:timeout_teacher.query(states[0],visits=visits)
            except TimeoutError:unknown=True
            else:raise RuntimeError('The controlled startup timeout fixture unexpectedly returned a label.')
            if timeout_teacher._active is not None:raise RuntimeError('Teacher process survived UNKNOWN timeout.')
        finally:timeout_teacher.close()
    finally:teacher.close()
    result={'status':'PASS','actual_gpu_predictions_exact':True,'query_order_independent':True,
        'states':2,'queries':8,'visits_per_query':visits,'forced_queries_included':True,
        'unknown_timeout_preserved':unknown,'gpu_device':gpu_device,'durations_seconds':durations,
        'protocol':go2_k20_stateless_teacher_identity(project),'assets':{k:v for k,v in go2_k20_gpu_assets(project).items() if k!='env'},
        'validated_at_utc':go2_now(),'scope':'Disposable offline teacher label queries; no model training or matches'}
    go2_atomic_json(project/'reports/ahr_v2/fixtures/k20/stateless-teacher-validation.json',result)
    return result


### Verified promotion and playable checkpoint

In [ ]:
import copy, io, dataclasses, hashlib, statistics, collections, uuid
GO2_K20_PROMOTION_POLICY={'min_kata_games':20,'min_kata_wins':1,'min_gnu_games':12,
    'min_gnu_win_rate':.70,'max_gnu_drop':.10,'max_unknown':0,
    'ahr_soft_move_seconds':2.0,'min_opening_plies':8}

def _go2_k20_frozen_champion(project,device='cpu'):
    run_id='ahr-v2-teacher-20261008';manifest=go2_teacher_candidate_manifest(project,run_id)
    entries=[entry for entry in manifest['entries'] if entry['optimizer_step']==8016]
    if len(entries)!=1:raise RuntimeError('Frozen8016 champion snapshot missing or ambiguous.')
    return go2_load_selected_teacher_player(project,run_id=run_id,candidate_file=entries[0]['file'],device=device)

def _go2_k20_report_path(project,path):
    result=Path(path);result=result if result.is_absolute() else Path(project)/result;result=result.resolve()
    if not result.is_relative_to(Path(project).resolve()/'reports') or not result.is_file():raise ValueError('Promotion evidence must be a saved project report.')
    return result

def _go2_k20_verified_report(project,path,profile,model_identity=None):
    path=_go2_k20_report_path(project,path);report=json.loads(path.read_text())
    if report.get('status')!='COMPLETE' or report.get('config',{}).get('purpose')!='development':raise RuntimeError('Only complete development arena evidence qualifies.')
    if model_identity is not None and report.get('identity',{}).get('model_identity')!=model_identity:raise RuntimeError('Evidence belongs to a different candidate model.')
    games=[game for game in report['games'] if game['profile']==profile]
    if len(games)!=len(report['games']) or not games:raise RuntimeError('The promotion report must contain only the requested opponent profile.')
    pin=report['identity']['opponents'][profile]
    current=go2_bench_opponent_pin(project,profile)
    if go2_digest(pin)!=go2_digest(current):raise RuntimeError('Promotion opponent pin differs from the benchmark reference.')
    for game in games:
        state=go2_initial_state(9,7.5)
        for item in game['trace']:
            if state.to_play!=item['color']:raise RuntimeError('Evidence trace has an invalid player.')
            state=go2_step(state,int(item['action']))
        if list(state.board)!=game['state']['board']:raise RuntimeError('Evidence board differs from its legal replay.')
        if game['outcome']=='UNKNOWN':
            if not game.get('unknown_reason'):raise RuntimeError('UNKNOWN evidence lacks its cutoff/error reason.')
            continue
        if not state.terminal or state.consecutive_passes!=2:raise RuntimeError('Evidence game is not a verified terminal replay.')
        score=go2_score(state)['score_black']*game['ahr_color'];expected='WIN' if score>0 else 'LOSS' if score<0 else 'DRAW'
        if expected!=game['outcome'] or score!=game['score_for_ahr']:raise RuntimeError('Evidence outcome/score does not match complete replay.')
        if game['opening_family']!=go2_teacher_opening_family(game['opening_actions']):raise RuntimeError('Evidence opening-family hash mismatch.')
    pairs=collections.Counter((game['opening_family'],game['ahr_color']) for game in games)
    if any(v!=1 for v in pairs.values()):raise RuntimeError('Repeated family/color pairs are not fresh confirmation.')
    families={game['opening_family'] for game in games}
    if any((family,1) not in pairs or (family,-1) not in pairs for family in families):raise RuntimeError('Every confirmation opening needs both AHR colors.')
    return {'path':str(path.relative_to(Path(project).resolve())),'sha256':go2_file_sha(path),
        'report':report,'games':games,'families':families,'pairs':pairs,
        'wins':sum(g['outcome']=='WIN' for g in games),'unknown':sum(g['outcome']=='UNKNOWN' for g in games)}

def _go2_k20_promotion_exclusions(project,evidence,metadata=None):
    project=Path(project);families=set();provenance=[]
    cache_ids=['go9x9-teacher-v1',evidence.get('training_cache_id','go9x9-competitive-v2')]
    for cache_id in dict.fromkeys(cache_ids):
        path=go2_teacher_cache_path(project,cache_id)/'manifest.json'
        if not path.is_file():raise RuntimeError('Training-family inventory missing for promotion: '+cache_id)
        manifest=json.loads(path.read_text())
        if cache_id!='go9x9-teacher-v1':
            verified=go2_k18_dataset(project,cache_id,verify=False)
            if metadata is not None:
                actual=metadata.get('data_identity')
                actual=actual.get('identity') if isinstance(actual,dict) else actual
                if actual is not None and actual!=verified['identity']:raise RuntimeError('Promotion training-family cache differs from candidate training data.')
        families.update(manifest['families']['train']);families.update(manifest['families']['dev'])
        provenance.append({'path':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    benchmark=project/'reports/ahr_v2/benchmarks/ahr-v2-benchmark-20261009-v1/report.json'
    if not benchmark.is_file():raise RuntimeError('Consumed benchmark17 inventory missing.')
    families.update(g['opening_family'] for g in json.loads(benchmark.read_text())['games'])
    provenance.append({'path':str(benchmark.relative_to(project)),'sha256':go2_file_sha(benchmark)})
    for family in evidence.get('additional_excluded_families',[]):
        if not isinstance(family,str) or len(family)!=64:raise ValueError('Invalid extra excluded family hash.')
        families.add(family)
    if metadata and metadata.get('model_type')=='Go2K20Network':
        path=project/'checkpoints'/metadata['run_id']/metadata['checkpoint_file']
        state=torch.load(path,map_location='cpu',weights_only=False);buffer_id=state['config']['buffer_id']
        buffer,entry=go2_load_checkpoint(project/'checkpoints'/(buffer_id+'-collection'))
        if buffer is None or buffer.get('stage')!='COMPLETE':raise RuntimeError('Guided training buffer family inventory incomplete.')
        families.update(buffer['sampler']['opening_families'])
        provenance.append({'path':str((project/'checkpoints'/(buffer_id+'-collection')/entry['file']).relative_to(project)),'sha256':entry['sha256']})
    return families,provenance

def _go2_k20_gate_counts(kata,gnu_candidate,gnu_champion,policy):
    reasons=[]
    if kata['games']<policy['min_kata_games']:reasons.append('Insufficient fresh KataGo1 confirmation games.')
    if kata['wins']<policy['min_kata_wins']:reasons.append('KataGo1 confirmed wins below the promotion threshold.')
    if max(kata['unknown'],gnu_candidate['unknown'],gnu_champion['unknown'])>policy['max_unknown']:reasons.append('UNKNOWN outcomes cannot qualify as evidence.')
    if min(gnu_candidate['games'],gnu_champion['games'])<policy['min_gnu_games']:reasons.append('GNU regression comparison is too small.')
    candidate_rate=gnu_candidate['wins']/max(1,gnu_candidate['games']);champion_rate=gnu_champion['wins']/max(1,gnu_champion['games'])
    if candidate_rate+1e-12<policy['min_gnu_win_rate']:reasons.append('Candidate GNU win rate below the floor.')
    if champion_rate-candidate_rate>policy['max_gnu_drop']+1e-12:reasons.append('GNU regression exceeds the allowed drop.')
    return {'decision':'PASS' if not reasons else 'KEEP_CHAMPION','reasons':reasons,
        'kata_wins':kata['wins'],'kata_games':kata['games'],'gnu_candidate_win_rate':candidate_rate,
        'gnu_champion_win_rate':champion_rate,'gnu_drop':champion_rate-candidate_rate,
        'evidence_scope':'Development promotion gate; a point estimate is not a confidence or ranking claim.'}

def go2_k20_promotion_decision(project,evidence,model_identity=None,policy=None,metadata=None):
    project=go2_project(project);policy=copy.deepcopy(GO2_K20_PROMOTION_POLICY)|copy.deepcopy(policy or {})
    if set(policy)!=set(GO2_K20_PROMOTION_POLICY):raise ValueError('Unknown promotion policy field.')
    if min(policy['min_kata_games'],policy['min_gnu_games'])<2 or policy['min_kata_wins']<1 or policy['max_unknown']!=0:
        raise ValueError('Promotion requires real KataGo wins and zero UNKNOWN games.')
    kata=_go2_k20_verified_report(project,evidence['katago_confirmation'],'katago-v1',model_identity)
    gnu=_go2_k20_verified_report(project,evidence['gnu_candidate'],'gnugo-l10',model_identity)
    old=_go2_k20_verified_report(project,evidence['gnu_champion'],'gnugo-l10')
    champion,champion_meta=_go2_k20_frozen_champion(project,device='cpu')
    if old['report']['identity']['model_identity']!=champion_meta['model_identity']:raise RuntimeError('GNU regression reference is not the frozen 8016 champion.')
    if gnu['pairs']!=old['pairs']:raise RuntimeError('GNU candidate and champion must share the same fresh opening/color pairs.')
    excluded,provenance=_go2_k20_promotion_exclusions(project,evidence,metadata)
    for tested in [kata,gnu,old]:
        if tested['families']&excluded:raise RuntimeError('Confirmation reused training/development/consumed families.')
        if any(len(g['opening_actions'])<policy['min_opening_plies'] for g in tested['games']):raise RuntimeError('Empty or short consumed openings are not fresh confirmation.')
        budget=tested['report']['config'].get('ahr_move_seconds')
        if budget is None or abs(float(budget)-policy['ahr_soft_move_seconds'])>1e-9:raise RuntimeError('Promotion move budget differs from the fixed benchmark target.')
    if kata['families']&gnu['families']:raise RuntimeError('Kata confirmation and GNU regression families must be separate.')
    counts=lambda item:{'games':len(item['games']),'wins':item['wins'],'unknown':item['unknown']}
    result=_go2_k20_gate_counts(counts(kata),counts(gnu),counts(old),policy)
    result.update({'policy':policy,'evidence_files':{name:{k:item[k] for k in ['path','sha256']} for name,item in [('katago_confirmation',kata),('gnu_candidate',gnu),('gnu_champion',old)]},
        'family_exclusions':provenance,'fresh_kata_families':len(kata['families']),'fresh_gnu_families':len(gnu['families']),
        'candidate_model_identity':model_identity,'champion_model_identity':champion_meta['model_identity'],'verified_at_utc':go2_now()})
    return result

def _go2_k20_snapshot_model(project,path,sha,model_type,device='cpu'):
    path=Path(path)
    if not path.is_file() or go2_file_sha(path)!=sha:raise RuntimeError('Promoted snapshot checksum mismatch.')
    state=torch.load(path,map_location='cpu',weights_only=False)
    if GO2_REQUIRED_STATE-set(state) or state['schema']!=GO2_SCHEMA:raise RuntimeError('Promoted snapshot is not a complete resumable checkpoint.')
    if model_type=='Go2K20Network':
        if state['identity'].get('schema')!='ahr-go-v2-guided-training-v1' or state['identity']['sources']!=go2_k20_sources(project):raise RuntimeError('Promoted notebook20 snapshot/source mismatch.')
        with torch.random.fork_rng(devices=[]):model=Go2K20Network(Go2NetConfig(**state['identity']['network']))
    elif model_type=='Go2Network':
        if state['identity'].get('training_schema')!=GO2_K19_SCHEMA or state['identity']['sources']!=go2_k19_sources(project):raise RuntimeError('Promoted notebook19 snapshot/source mismatch.')
        with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(board_size=9,**state['config']['network']))
    else:raise RuntimeError('Unsupported promoted model type.')
    model.load_state_dict(state['model'],strict=True);model.to(device).eval();model.requires_grad_(False)
    return model,state

def _go2_k20_read_promotion_pointer(project,path=None):
    project=Path(project);path=Path(path or project/'reports/ahr_v2/k20_promoted_player.json')
    if not path.exists():return None
    value=json.loads(path.read_text());signed=dict(value);identity=signed.pop('pointer_sha256',None)
    if value.get('schema')!='ahr_k20_promoted_player_v1' or identity!=go2_digest(signed):raise RuntimeError('Promoted player pointer is corrupt; no silent fallback.')
    snapshot=(project/value['snapshot_file']).resolve()
    if not snapshot.is_relative_to((project/'checkpoints').resolve()) or go2_file_sha(snapshot)!=value['snapshot_sha256']:raise RuntimeError('Promoted snapshot path/checksum mismatch.')
    return value

def go2_k20_promote(project,pipeline_id,model,metadata,evidence,policy=None):
    project=go2_project(project)
    if Path(pipeline_id).name!=pipeline_id:raise ValueError('Local pipeline ID required.')
    if metadata.get('is_smoke') or metadata.get('optimizer_step',0)<=0:raise RuntimeError('Disposable or untrained models cannot be promoted.')
    model_identity=go2_model_identity(model)
    if model_identity!=metadata['model_identity']:raise RuntimeError('Candidate model differs from checkpoint metadata.')
    result=go2_k20_promotion_decision(project,evidence,model_identity,policy,metadata)
    result['promoted']=False
    report_path=project/'reports/ahr_v2'/pipeline_id/'promotion.json';go2_atomic_json(report_path,result)
    if result['decision']!='PASS':return result
    directory=project/'checkpoints/ahr-v2-promoted-players';source=(project/'checkpoints'/metadata['run_id']/metadata['checkpoint_file']).resolve()
    if not source.is_relative_to((project/'checkpoints'/metadata['run_id']).resolve()):raise ValueError('Candidate checkpoint path escapes its run.')
    model_type='Go2K20Network' if metadata.get('model_type')=='Go2K20Network' else 'Go2Network'
    verified,state=_go2_k20_snapshot_model(project,source,metadata['checkpoint_sha256'],model_type,'cpu')
    if go2_model_identity(verified)!=model_identity:raise RuntimeError('Candidate checkpoint weights differ from the evaluated model.')
    with go2_checkpoint_lock(directory):
        # Detect a corrupt existing pointer before replacing it.
        _go2_k20_read_promotion_pointer(project)
        snapshot=directory/('snapshot-'+model_identity[:24]+'.pt')
        if snapshot.exists() and go2_file_sha(snapshot)!=metadata['checkpoint_sha256']:raise RuntimeError('Promoted snapshot identity collision.')
        if not snapshot.exists():go2_atomic_bytes(snapshot,source.read_bytes())
        if go2_file_sha(snapshot)!=metadata['checkpoint_sha256']:raise RuntimeError('Promoted checkpoint copy failed hash verification.')
        value={'schema':'ahr_k20_promoted_player_v1','pipeline_id':pipeline_id,'model_type':model_type,
            'model_identity':model_identity,'snapshot_file':str(snapshot.relative_to(project)),
            'snapshot_sha256':metadata['checkpoint_sha256'],'metadata':copy.deepcopy(metadata),'evidence':result,
            'promoted_at_utc':go2_now()}
        value['pointer_sha256']=go2_digest(value);go2_atomic_json(project/'reports/ahr_v2/k20_promoted_player.json',value)
    published=result|{'promoted':True,'pointer':'reports/ahr_v2/k20_promoted_player.json'}
    go2_atomic_json(report_path,published);return published

def go2_k20_load_promoted_player(project,device='cpu',allow_smoke=False):
    project=go2_project(project);pointer=_go2_k20_read_promotion_pointer(project)
    if pointer is None:
        model,metadata=_go2_k20_frozen_champion(project,device=device)
        metadata=copy.deepcopy(metadata);metadata.update(label='Current AHR champion — frozen 8016; no newer promotion',promotion_status='KEEP_CHAMPION',model_type='Go2Network')
        return model,metadata
    model,state=_go2_k20_snapshot_model(project,project/pointer['snapshot_file'],pointer['snapshot_sha256'],pointer['model_type'],device)
    metadata=copy.deepcopy(pointer['metadata'])
    if metadata.get('is_smoke') and not allow_smoke:raise RuntimeError('Disposable checkpoint cannot be a human opponent.')
    if go2_model_identity(model)!=pointer['model_identity']:raise RuntimeError('Promoted model identity mismatch.')
    metadata.update(label='Promoted AHR challenger — '+metadata['run_id'],promotion_status='PASS',model_type=pointer['model_type'],
        best_selection_basis=f"Fresh KataGo1 confirmation {pointer['evidence']['kata_wins']}/{pointer['evidence']['kata_games']}; GNU regression passed; development evidence")
    return model,metadata


In [ ]:
def go2_k20_play_namespace(project):
    project=go2_project(project);namespace=go2_k18_namespace(project)
    namespace['go2_import_exports'](project/'experiments/ahr_v2/19_stronger_scout_worker_training.ipynb',namespace)
    namespace['go2_import_exports'](project/'experiments/ahr_v2/20_economic_rewards_and_teacher_guided_improvement.ipynb',namespace)
    base=namespace['Go2TeacherPlaySession'];base_ui=namespace['Go2TeacherPlayUI']
    source_fn=lambda p:go2_k20_sources(p)
    namespace['go2_teacher_play_sources']=source_fn
    namespace['go2_teacher_play_load_model']=lambda p,which='best',device='cpu',allow_smoke=False:go2_k20_load_promoted_player(p,device,allow_smoke)
    def factory(config,kind):
        cls=namespace['Go2K20Network'] if kind=='Go2K20Network' else namespace['Go2Network']
        with torch.random.fork_rng(devices=[]):return cls(namespace['Go2NetConfig'](**config))
    class Session(base):
        def __init__(self,project,model,metadata,human_color=1,search_config=None,game_id=None,persist=True):
            config=search_config or namespace['Go2SearchConfig'](simulations=512,batch_size=8,wall_time_seconds=2.0)
            super().__init__(project,model,metadata,human_color,config,game_id,persist)
            self.directory=self.project/'checkpoints/ahr-v2-katago-human-games'/self.game_id
        def save(self):
            if not self._persist:return
            self.directory=self.project/'checkpoints/ahr-v2-katago-human-games'/self.game_id
            namespace['go2_teacher_play_verify_replay'](self.state)
            if source_fn(self.project)!=self.sources:raise RuntimeError('Human player source definitions changed during this game.')
            path=self.directory/'game.json'
            with namespace['go2_checkpoint_lock'](self.directory):
                if path.exists() and json.loads(path.read_text()).get('revision')!=self.revision:raise RuntimeError('Another session changed this game.')
                if not path.exists() and self.revision:raise RuntimeError('Saved game disappeared; no silent restart.')
                snapshot=self.directory/'opponent.pt'
                if not snapshot.exists():
                    payload={'schema':'ahr_k20_human_snapshot_v1','model_type':self.metadata.get('model_type','Go2Network'),
                        'config':dataclasses.asdict(self.model.config),'model':{k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}}
                    stream=io.BytesIO();torch.save(payload,stream);go2_atomic_bytes(snapshot,stream.getvalue())
                record={'schema':'ahr_k20_human_game_v1','game_id':self.game_id,'revision':self.revision+1,'sources':self.sources,
                    'model_identity':self.metadata['model_identity'],'snapshot_file':'opponent.pt','snapshot_sha256':go2_file_sha(snapshot),
                    'metadata':self.metadata,'human_color':self.human_color,'search_config':dataclasses.asdict(self.search_config),
                    'state':namespace['go2_state_to_dict'](self.state),'state_sha256':namespace['go2_state_digest'](self.state),
                    'events':self.events,'last_search':self.last_search,'updated_at_utc':go2_now()}
                record=json.loads(json.dumps(record,default=go2_json_default,allow_nan=False));record['payload_sha256']=go2_digest(record)
                go2_atomic_json(path,record);self.revision=record['revision']
                if not self.metadata.get('is_smoke'):go2_atomic_json(self.project/'reports/ahr_v2/k20_latest_human_game.json',{'game_path':str(path.relative_to(self.project)),'updated_at_utc':record['updated_at_utc']})
        @classmethod
        def restore(cls,project,game_path,device='cpu'):
            project=Path(project);path=Path(game_path);path=path if path.is_absolute() else project/path;path=path.resolve()
            if not path.is_relative_to((project/'checkpoints/ahr-v2-katago-human-games').resolve()) or path.name!='game.json':raise ValueError('Resume a saved notebook20 AHR game.')
            record=json.loads(path.read_text());checksum=record.pop('payload_sha256',None)
            if checksum!=go2_digest(record) or record.get('schema')!='ahr_k20_human_game_v1':raise RuntimeError('Human game record checksum/schema mismatch.')
            if record['sources']!=source_fn(project):raise RuntimeError('Human-game sources changed; resume refused.')
            snapshot=path.parent/'opponent.pt'
            if record['snapshot_file']!='opponent.pt' or go2_file_sha(snapshot)!=record['snapshot_sha256']:raise RuntimeError('Frozen human opponent snapshot corrupt.')
            payload=torch.load(snapshot,map_location='cpu',weights_only=True)
            if payload['schema']!='ahr_k20_human_snapshot_v1':raise RuntimeError('Frozen human opponent schema mismatch.')
            model=factory(payload['config'],payload['model_type']);model.load_state_dict(payload['model'],strict=True);model.to(device).eval();model.requires_grad_(False)
            if go2_model_identity(model)!=record['model_identity']:raise RuntimeError('Frozen human opponent model identity differs.')
            state=namespace['go2_state_from_dict'](record['state']);namespace['go2_teacher_play_verify_replay'](state)
            if namespace['go2_state_digest'](state)!=record['state_sha256']:raise RuntimeError('Saved human Go history differs.')
            session=cls(project,model,record['metadata'],human_color=record['human_color'],search_config=namespace['Go2SearchConfig'](**record['search_config']),game_id=record['game_id'],persist=False)
            session.directory=path.parent;session.state=state;session.events=record['events'];session.last_search=record['last_search'];session.revision=record['revision'];session._persist=True
            return session
    namespace['Go2TeacherPlaySession']=Session
    class UI(base_ui):
        def __init__(self,*args,**kwargs):
            project=Path(args[0] if args else kwargs['project'])
            path=project/'reports/ahr_v2/k20_latest_human_game.json'
            previous_game=json.loads(path.read_text())['game_path'] if path.exists() else None
            super().__init__(*args,**kwargs)
            self.resume_path=previous_game
            self.which.layout.display='none';self.which.description='Verified player';self.visits.value=512
            self.buttons['Reload checkpoint'].description='Reload verified player';self.buttons['Reload checkpoint'].layout.width='190px'
            self.widget.children[0].value='<h3 style="margin:0">Play verified AHR · 9×9</h3>'
    namespace['Go2TeacherPlayUI']=UI
    return namespace

def go2_k20_play(project,device='cpu',persist=True,display_widget=True):
    project=go2_project(project);model,metadata=go2_k20_load_promoted_player(project,device)
    namespace=go2_k20_play_namespace(project);ui=namespace['Go2TeacherPlayUI'](project,model,metadata,device=device,persist=persist)
    if display_widget:
        from IPython.display import display
        display(ui.widget)
    return ui

def go2_k20_promotion_play_validation(project):
    project=go2_project(project);policy=copy.deepcopy(GO2_K20_PROMOTION_POLICY)
    kata={'games':20,'wins':1,'unknown':0};gnu={'games':20,'wins':16,'unknown':0}
    if _go2_k20_gate_counts(kata,gnu,gnu,policy)['decision']!='PASS':raise AssertionError('Qualifying fixture evidence failed.')
    if _go2_k20_gate_counts(kata|{'wins':0},gnu,gnu,policy)['decision']!='KEEP_CHAMPION':raise AssertionError('Zero wins allowed promotion.')
    if _go2_k20_gate_counts(kata|{'unknown':1},gnu,gnu,policy)['decision']!='KEEP_CHAMPION':raise AssertionError('UNKNOWN allowed promotion.')
    if _go2_k20_gate_counts(kata,gnu|{'wins':10},gnu,policy)['decision']!='KEEP_CHAMPION':raise AssertionError('GNU regression allowed promotion.')
    namespace=go2_k20_play_namespace(project);kind='Go2K20Network'
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(202020);model=namespace[kind](namespace['Go2NetConfig'](board_size=9,width=16,blocks=1))
    model.eval();model.requires_grad_(False);identity=go2_model_identity(model)
    metadata={'model_identity':identity,'model_type':kind,'is_smoke':True,'optimizer_step':1,'label':'Disposable auxiliary-head restoration fixture','run_id':'k20-fixture'}
    session=namespace['Go2TeacherPlaySession'](project,model,metadata,game_id='k20-play-fixture-'+uuid.uuid4().hex[:10],persist=True)
    session.move(39);session.human_color=0;session.move(41,actor='ahr')
    restored=namespace['Go2TeacherPlaySession'].restore(project,session.directory/'game.json')
    if go2_tensor_digest(restored.model.state_dict())!=go2_tensor_digest(model.state_dict()):raise AssertionError('Auxiliary model heads were not restored exactly.')
    if go2_state_digest(restored.state)!=go2_state_digest(session.state):raise AssertionError('Human game history did not restore exactly.')
    ui=go2_k20_play(project,persist=False,display_widget=False)
    current_pointer=_go2_k20_read_promotion_pointer(project)
    expected_identity=current_pointer['model_identity'] if current_pointer else _go2_k20_frozen_champion(project)[1]['model_identity']
    if len(ui.points)!=81 or ui.visits.value!=512 or ui.which.layout.display!='none' or ui.metadata['model_identity']!=expected_identity:raise AssertionError('Default frozen-player human UI is not ready.')
    fixture=project/'reports/ahr_v2/fixtures'/('k20-promotion-pointer-'+uuid.uuid4().hex[:10]+'.json')
    snapshot=session.directory/'opponent.pt'
    pointer={'schema':'ahr_k20_promoted_player_v1','snapshot_file':str(snapshot.relative_to(project)),
        'snapshot_sha256':go2_file_sha(snapshot),'purpose':'validation_fixture'}
    pointer['pointer_sha256']=go2_digest(pointer);go2_atomic_json(fixture,pointer)
    if _go2_k20_read_promotion_pointer(project,fixture) is None:raise AssertionError('Valid pointer fixture failed.')
    pointer['snapshot_sha256']='0'*64;pointer['pointer_sha256']=go2_digest({k:v for k,v in pointer.items() if k!='pointer_sha256'});go2_atomic_json(fixture,pointer)
    try:_go2_k20_read_promotion_pointer(project,fixture)
    except RuntimeError:pass
    else:raise AssertionError('Snapshot mismatch was accepted.')
    pointer['snapshot_sha256']=go2_file_sha(snapshot);pointer['pointer_sha256']='0'*64;go2_atomic_json(fixture,pointer)
    try:_go2_k20_read_promotion_pointer(project,fixture)
    except RuntimeError:pass
    else:raise AssertionError('Pointer corruption was accepted.')
    receipt={'schema':'ahr_k20_promotion_play_validation_v1','status':'PASS','zero_wins_rejected':True,
        'unknown_rejected':True,'gnu_regression_rejected':True,'auxiliary_heads_exact_restore':True,
        'human_history_exact_restore':True,'champion_human_ui_ready':True,'pointer_and_snapshot_corruption_rejected':True,
        'production_pointer_changed':False,'source_identity':go2_k20_sources(project),'validated_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/k20_promotion_play_validation/receipt.json',receipt);return receipt


## Run the complete workflow

**Current run (12 October):** training and the 100-game KataGo confirmation are complete. The selected model won **3/100** against KataGo at one visit, below the **55/100** promotion threshold. The remaining GNU Go regression games use a validated evaluation-recovery notebook because the original profile filter retained nonconsecutive game indices. Run All now resumes only those remaining evaluation stages. The original training, checkpoints, paired openings and completed KataGo matches are preserved.

Recovery checkpoints: `checkpoints/ahr-v2-katago-econ-20261011-v1-eval-recovery-v1*/`. Recovery definitions: `reports/ahr_v2/ahr-v2-katago-econ-20261011-v1/evaluation_recovery.ipynb`. Expect roughly 1–2 hours for the 40 remaining GNU Go games plus speed checks, depending on game length. The same tmux attach/detach and Run All resume controls below apply.

The original workflow prepared two million training positions, compared a warm-started 96×6 network with a fresh 128×10 network, collected AHR games, and trained matched control/economics variants. These stages are complete; their artifacts remain available below. KataGo supplies offline labels and plays evaluation games; AHR chooses its own moves during human play.

The supervised stages each have a three-hour cap and a 20,000-update limit. Collection and guided stages have separate eight-hour caps; they stop earlier when their registered work completes. Match evaluation adds time. These are caps, not an estimate or a claim that the GPU remains busy for that duration. Inspect the recorded active time, row count and optimizer steps.

Checkpoints live under `checkpoints/ahr-v2-katago-econ-20261011-v1/` for the controller, `checkpoints/ahr-k19-20261011-{control,challenger}/` for supervised training, and `checkpoints/ahr-v2-katago-econ-20261011-v1-{buffer-collection,guided-control,guided-economics}/` for collection and improvement. Arena stages have their own checkpoint directories. Each has a single writer and a strict manifest.

For exact resume, keep the IDs, configuration, data and tagged definition cells unchanged and click **Run All**. It detects a live tmux session or resumes the verified stage and substage checkpoint. A corrupt or mismatched checkpoint raises an error instead of silently starting over. After an intentional stop, set `RESUME_STOPPED_RUN=True` to resume.

Attach with `tmux attach -t ahr-v2-katago-econ20`; detach with **Ctrl-b**, then **d**. The run survives a browser or SSH disconnect while the GPU instance stays on. This workspace is on the persistent boot disk; the deletion policy is unverified and the off-instance backup remains deferred at your request.

Final/test data stays unopened. Fresh paired development matches are excluded from training. Promotion requires at least 55 wins from 100 fresh KataGo 1-visit games, zero UNKNOWN, and a GNU Go level10 regression check against the frozen incumbent. A 1-visit result applies to that limited opponent budget, not unrestricted KataGo. Passing stays a legal move; no stone-capture or survival bounty replaces winning.

The economics idea is **estimated opportunity cost**: give bounded extra learning weight to decisions that lose teacher-estimated winning probability. A separate credit head learns outcomes relative to a frozen forecast. The main Worker continues to estimate unshaped win/draw/loss. Potential-difference reward accounting cancels around closed paths and at the terminal boundary; it does not guarantee better learning. The paired control experiment measures the effect.

Primary method references: [potential-based shaping](https://ai.stanford.edu/~ang/papers/shaping-icml99.pdf), [stochastic-game extension](https://arxiv.org/abs/1401.3907), [DAgger](https://proceedings.mlr.press/v15/ross11a.html), and [Expert Iteration](https://arxiv.org/abs/1705.08439).

In [ ]:
PIPELINE_MODE=os.environ.get('AHR_K20_MODE','run')
PIPELINE_CONFIG=go2_k20_pipeline_config()
TMUX_SESSION='ahr-v2-katago-econ20'
RESUME_STOPPED_RUN=False
RECOVERY_BOOK=PROJECT/'reports/ahr_v2/ahr-v2-katago-econ-20261011-v1/evaluation_recovery.ipynb'
USE_EVALUATION_RECOVERY=RECOVERY_BOOK.exists()
if USE_EVALUATION_RECOVERY:
    go2_import_exports(RECOVERY_BOOK,globals())
if PIPELINE_MODE not in ('run','validate','status'):
    raise ValueError('AHR_K20_MODE must be run, validate, or status.')
if PIPELINE_MODE=='run':
    if USE_EVALUATION_RECOVERY:
        LAUNCH=go2_k20_recovery_launch(PROJECT,session=TMUX_SESSION,resume_stopped=RESUME_STOPPED_RUN)
    else:
        LAUNCH=go2_k20_pipeline_launch(PROJECT,PIPELINE_CONFIG,session=TMUX_SESSION,resume_stopped=RESUME_STOPPED_RUN)
    display(Markdown('**'+LAUNCH['status']+'** · session `'+TMUX_SESSION+'`'))
    display(Markdown('Attach: `tmux attach -t '+TMUX_SESSION+'` · detach: **Ctrl-b**, then **d**.'))
else:
    display(Markdown('**'+PIPELINE_MODE+' mode:** the main pipeline is not launched by this execution.'))


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
progress_output=widgets.Output()
refresh_pipeline=widgets.Button(description='Refresh progress',icon='refresh')
stop_pipeline=widgets.Button(description='Request checkpointed stop',icon='stop',button_style='warning')
def show_pipeline(_=None):
    with progress_output:
        clear_output(wait=True)
        status=go2_k20_recovery_status(PROJECT) if USE_EVALUATION_RECOVERY else go2_k20_pipeline_status(PROJECT,PIPELINE_CONFIG['pipeline_id'])
        display(Markdown('**Pipeline:** '+status['status']+' · **Stage:** '+str(status.get('stage','—'))))
        if status.get('updated_at_utc'):
            display(Markdown('Updated '+go2_bench_dublin(status['updated_at_utc'])))
        stages=status.get('completed_stages',[])
        if stages:display(Markdown('Completed: '+', '.join(stages)))
        report_root=PROJECT/'reports/ahr_v2'/PIPELINE_CONFIG['pipeline_id']
        if (report_root/'results.json').exists():
            saved=json.loads((report_root/'results.json').read_text())['results']
            rows=[]
            for key in ('train_control','train_challenger','guided_control','guided_economics'):
                item=saved.get(key)
                if item:
                    rows.append({'stage':key,'optimizer steps':item.get('optimizer_steps',item.get('global_step')),
                        'active minutes':round(item['active_seconds']/60,1) if 'active_seconds' in item else None})
            if rows:
                import pandas as pd
                display(pd.DataFrame(rows))
            if 'economic_screen' in saved:
                rows=[{'candidate':k,'KataGo1 wins':v['summary']['WIN'],'games':v['summary']['attempts'],'UNKNOWN':v['summary']['UNKNOWN']}
                    for k,v in saved['economic_screen']['candidates'].items()]
                display(pd.DataFrame(rows))
                display(Markdown('Same paired openings; this development comparison alone does not establish a general strength advantage.'))
            if 'promotion_and_report' in saved:
                display(Markdown('**Promotion:** '+str(saved['promotion_and_report'].get('decision',saved['promotion_and_report'].get('reason')))))
        if USE_EVALUATION_RECOVERY:
            display(Markdown('**Training complete. Fresh KataGo confirmation: 3 wins / 100 games, 0 UNKNOWN.** Promotion requires 55 wins; the frozen incumbent remains the playing model.'))
            recovery_id=GO2_K20_RECOVERY_PARENT+'-eval-recovery-v1'
            gnu_rows=[]
            for kind in ('candidate','incumbent'):
                path=PROJECT/'reports/ahr_v2/benchmarks'/(recovery_id+'-gnu-'+kind)/'report.json'
                if path.exists():
                    arena=json.loads(path.read_text());summary=arena['summary']
                    gnu_rows.append({'GNU Go level 10':kind,'status':arena['status'],'wins':summary['WIN'],
                        'completed / 20':summary['attempts'],'UNKNOWN':summary['UNKNOWN']})
            if gnu_rows:
                import pandas as pd
                display(pd.DataFrame(gnu_rows))
            path=PROJECT/'reports/ahr_v2'/recovery_id/'results.json'
            if path.exists():
                result=json.loads(path.read_text())['results'].get('promotion_and_report')
                if result:display(Markdown('**Promotion:** '+str(result.get('decision',result.get('reason')))))
        if status.get('error'):display(Markdown('**Stopped on error:** `'+status['error']+'`'))
def request_pipeline_stop(_):
    if USE_EVALUATION_RECOVERY:go2_k20_recovery_request_stop(PROJECT)
    else:go2_k20_pipeline_request_stop(PROJECT,PIPELINE_CONFIG)
    with progress_output:display(Markdown('**Stop requested.** The active stage will save its checkpoint. Resume requires `RESUME_STOPPED_RUN=True`.'))
refresh_pipeline.on_click(show_pipeline);stop_pipeline.on_click(request_pipeline_stop)
display(widgets.VBox([widgets.HBox([refresh_pipeline,stop_pipeline]),progress_output]))
show_pipeline()


## Play the verified player

This panel uses the promoted candidate if it passed the gate. Until then, it loads the frozen incumbent. It does not use KataGo to choose AHR moves. The saved game snapshot records which model you played; use the model reload control after a successful promotion.

In [ ]:
from IPython.display import HTML
# Presentation only: keep each playable intersection visible in notebook/widget renderers.
display(HTML('''<style>
.ahr-go-point.widget-button {
    margin: 0 !important; border: 0 !important; border-radius: 0 !important;
    background-image: linear-gradient(#76522d,#76522d), linear-gradient(#76522d,#76522d) !important;
    background-size: 100% 1px, 1px 100% !important;
    background-position: center !important; background-repeat: no-repeat !important;
}
</style>'''))
PLAY_UI=go2_k20_play(PROJECT,device='cpu',persist=True)
display(Markdown('**Playing model:** `'+PLAY_UI.session.metadata['run_id']+'` · optimizer step '+str(PLAY_UI.session.metadata.get('optimizer_step','—'))))